# 12 — Les grands modèles face à gemma4, à nos OCR et au PMQ : quel lecteur, avec quelle consigne ?

**La question :** des modèles **plus gros et meilleurs que gemma4** (les phares du marché, les grands modèles ouverts,
les OCR spécialisés) lisent-ils mieux les **noms et prénoms manuscrits** des bulletins du recensement ? Avec quelle
**consigne** ? À quel **coût** ? Et l'un d'eux fait-il **aussi bien que le PMQ** ?

**La méthode :** celle du notebook 10. Les mêmes bulletins, les mêmes images, la même consigne pour tous, jugés avec la
**vérité arbitrée par le SeRN**, et comparés au **PMQ** et à **gemma4** par des tests statistiques sur les mêmes lignes.

## Comment lancer

1. **Cellule 1** : vérifier les réglages. `TEST_RAPIDE = True` d'abord (10 bulletins).
2. **Lancer tout, de haut en bas.** L'essai (cellule 7) écarte les noms de modèles faux sans rien coûter.
3. Si tout est bon : `TEST_RAPIDE = False`, et relancer tout (les lectures déjà faites ne sont jamais repayées).
4. **Lire, dans l'ordre :** cellule 21 (classement et phrases), cellule 13 (tableau principal), cellule 14 (effet de
   la consigne), cellule 18 (barre du PMQ), cellule 19 (nos OCR).

## Le plan

| Cellule | Ce qu'elle fait | Coût Eden AI |
|---|---|---|
| 1, 1b | Réglages ; les consignes ; le journal des décisions | — |
| 2 – 4 | Divergences du SeRN, bulletins, Fiqual, blocs, clés | — |
| 5 | Lecture commune (parallèle, cache, temps, coût, plafond) | — |
| 7 | **Essai : 3 bulletins par modèle** | très faible |
| 8 | gemma4, avec chaque consigne (lectures déjà faites reprises) | aucun |
| 9 | Grands modèles × consignes, sur tous les bulletins | **principal** |
| 10 | OCR spécialisés | faible |
| 11 – 12 | Listes de l'Insee, alerte « lettre double », table ligne par ligne | — |
| 13 |  **Tableau principal** : chaque lecteur seul, + alerte, + désaccord avec gemma4 | — |
| 14 |  **Effet de la consigne** (V1 « épeler » contre V0 « simple »), modèle par modèle | — |
| 15 – 17 | Groupes A/B/C, divergences PMQ / PAD, types d'erreurs | — |
| 18 |  **La barre du PMQ** | — |
| 19 | **Nos OCR** (TrOCR, PyLaia, TeleOCR) sur les mêmes lignes | — |
| 20 | Temps, coût, projection annuelle | — |
| 21 |  **Classement final et phrases pour la réunion** | — |
| 22 – 23 | Excel détaillé, et **MLflow : tout le détail** (réglages, mesures, tableaux, lectures, Excel) | — |

Le **journal des décisions**, en fin de notebook, justifie chaque choix de méthode.

In [ ]:
# 1. REGLAGES : tout ce qu'on peut changer est ici (le reste du notebook ne se modifie pas)
import base64
import contextlib
import getpass
import io
import json
import math
import os
import re
import threading
import time
import unicodedata
import zipfile
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from pathlib import Path

import mlflow
import pandas as pd
import requests
import s3fs
from PIL import Image
from scipy import stats

CAMPAGNE = "RG25"
TEST_RAPIDE = False                  # True d'abord : 10 bulletins pour tout verifier ; puis False (169 bulletins)
N_ESSAI = 169                       # etape 7 : bulletins par modele pour l'essai (avant toute depense)

# ---- 1) LES GRANDS MODELES QUI VOIENT LES IMAGES (VLM), ranges par categorie, avec la raison du choix ----
# Noms a confirmer dans Eden AI > Models : un nom faux est simplement ECARTE a l'essai (aucun cout).
MODELES_EDEN = {
    # A. Phares fermes : le haut de gamme du marche = le PLAFOND de ce qu'on peut esperer
    "google/gemini-pro-latest":       "A. phare ferme - Gemini Pro : tres fort en ecriture manuscrite et documents scannes",
    "openai/gpt-latest":              "A. phare ferme - modele phare d'OpenAI (nom donne par la documentation Eden AI)",
    "anthropic/claude-opus-latest":   "A. phare ferme - Claude Opus : suit bien les consignes, avoue plutot qu'inventer",
    # B. Ouverts PLUS GROS que gemma4 : installables un jour en interne = la piste PRODUCTION
    "qwen/qwen3-vl-plus":             "B. ouvert plus gros - Qwen-VL : tres fort en OCR ; notre qwen3-vl : 99 % sur les cases",
    "meta/llama-4-maverick":          "B. ouvert plus gros - Llama 4 : multimodal a poids ouverts",
    "mistral/pixtral-large-latest":   "B. ouvert plus gros - Pixtral (Mistral, francais) : vision a poids ouverts",
    # C. Rapides et economiques : le rapport qualite / prix
    "google/gemini-flash-latest":     "C. rapide et economique - Gemini Flash (deja teste, fonctionne)",
    "anthropic/claude-sonnet-latest": "C. rapide et economique - Claude Sonnet (nom donne par la documentation Eden AI)",
}
MODELES_RETENUS = []                # vide = automatiquement ceux qui reussissent l'essai

# ---- 2) LES OCR SPECIALISES (Expert Models d'Eden AI) : lecture litterale, sans consigne ----
OCR_EDEN = [
    "ocr/ocr/mistral",              # nom donne par Eden AI ; traite en Europe ; ~4 $ / 1000 pages
    "ocr/ocr/google",               # a confirmer dans le catalogue
    "ocr/ocr/microsoft",            # a confirmer dans le catalogue
]

# ---- 3) LES CONSIGNES testees (texte complet en cellule 1b) : chaque modele est lu avec chacune ----
CONSIGNES_TESTEES = ["V0", "V1"]    # V0 = simple (notebook 10) ; V1 = epeler lettre par lettre (ameliorer_consigne)

# ---- 4) NOS OCR deja calcules (TrOCR, PyLaia, TeleOCR) : compares sans aucun nouvel appel ----
COMPARER_NOS_OCR = True

# ---- 5) Garde-fous ----
PLAFOND_DOLLARS = 60.0              # arret des lectures Eden AI si le cout cumule (renvoye par Eden AI) depasse ce montant
MAX_JETONS = 2000                  # large : les modeles « qui reflechissent » consomment des jetons avant de repondre
COUVERTURE_MIN = 0.95               # un lecteur qui n'a pas lu 95 % des lignes est ecarte du tableau (comparaison equitable)
N_PARALLELE = 4                     # appels a la fois vers gemma4 (SSP Cloud)
N_PARALLELE_EDEN = 8                # appels a la fois vers Eden AI : les grands modeles sont lents, on en lance plus
DUREE_MAX_MINUTES = 60              # par modele et consigne : au-dela, arret propre (ce qui est lu est garde,
                                    # relancer reprend la ou on s'est arrete). Evite une cellule qui tourne des heures.
ENREGISTRER_TOUT = True             # MLflow : TOUT le detail (reglages, mesures, tableaux, lectures, Excel detaille).
                                    # Les fichiers restent dans le stockage S3 du projet, comme les donnees du banc.

MODELE_GEMMA = "gemma4-26b-moe"
URL_SSP = "https://llm.lab.sspcloud.fr/api/chat/completions"
URL_EDEN = "https://api.edenai.run/v3/chat/completions"
URL_UPLOAD = "https://api.edenai.run/v3/upload"
URL_UNIVERSAL = "https://api.edenai.run/v3/universal-ai"
DOCUMENTS_PAR_AN = 15 * 2450        # ordre de grandeur annuel du PMQ (15 lots de 2 450), pour projeter le cout

BUCKET = "projet-mesure-qualite-rp"
FIQUAL = {"RG25": "RP_PMQ_FIQUAL2599.txt", "RG26": "RP_PMQ_FIQUAL2699.txt"}
VARIABLE_NOM, VARIABLE_PRENOM = None, None     # None = detection automatique (colonne VARIABLE des divergences)
POS_UT, POS_CAB = 2, 4
COLS = {"NOM": 11, "PRENOM": 12}
BORDS_BLOC = (0.075, 0.072, 0.450, 0.105)
g, h, d_, b_ = BORDS_BLOC
ZONE_BLOC = (g, h, d_ - g, b_ - h)
RATIO_DOUBLE = 3                    # alerte lettre double : variante au moins 3 fois plus portee dans les listes

RESULTATS = Path.home() / "work" / "banc-htr" / "resultats" / "eden"
RESULTATS_10 = Path.home() / "work" / "banc-htr" / "resultats"       # lectures deja faites (notebooks 10, consigne)
LISTES = Path.home() / "work" / "banc-htr" / "listes_insee"
BLOCS = Path("/tmp/eden/blocs")
for d in [RESULTATS, BLOCS]:
    d.mkdir(parents=True, exist_ok=True)
SUFFIXE = "_rapide" if TEST_RAPIDE else ""


def normaliser(t):
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return ""
    t = unicodedata.normalize("NFKD", str(t).strip())
    t = "".join(c for c in t if not unicodedata.combining(c)).upper()
    t = re.sub(r"[^A-Z0-9 ]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def distance(a, b):
    if a == b:
        return 0
    prec = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cour = [i]
        for j, cb in enumerate(b, 1):
            cour.append(min(prec[j] + 1, cour[j - 1] + 1, prec[j - 1] + (ca != cb)))
        prec = cour
    return prec[-1]


def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, c - m), min(1.0, c + m)


def ic(k, n):
    lo, hi = wilson(k, n)
    return f"[{lo * 100:.1f} ; {hi * 100:.1f}]"


def mcnemar(a_ok, b_ok):
    """(lignes justes pour a seul, justes pour b seul, p exact)."""
    n10, n01 = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    return n10, n01, (stats.binomtest(n10, n10 + n01).pvalue if n10 + n01 else 1.0)


def manquant(v):
    return v is None or (isinstance(v, float) and math.isnan(v))


def lire_vault(chemin):
    """Secret Vault (KV v2) par l'API HTTP (la commande vault n'existe pas sur tous les services). {} si echec."""
    adresse, jeton = os.environ.get("VAULT_ADDR"), os.environ.get("VAULT_TOKEN")
    if not adresse or not jeton:
        return {}
    montage, reste = chemin.split("/", 1)
    try:
        r = requests.get(f"{adresse.rstrip('/')}/v1/{montage}/data/{reste}",
                         headers={"X-Vault-Token": jeton}, timeout=30)
        return r.json()["data"]["data"] if r.status_code == 200 else {}
    except Exception:
        return {}


CATEGORIE = {MODELE_GEMMA: "R. reference interne - gemma4 26B (SSP Cloud), gratuit", **MODELES_EDEN,
             **{o: "D. OCR specialise - " + o.split("/")[-1] + " : lecture litterale, sans consigne" for o in OCR_EDEN}}
print(f"campagne {CAMPAGNE} | mode {'TEST RAPIDE (10 bulletins)' if TEST_RAPIDE else 'COMPLET'} | "
      f"{len(MODELES_EDEN)} VLM x {len(CONSIGNES_TESTEES)} consignes, {len(OCR_EDEN)} OCR | plafond {PLAFOND_DOLLARS} $")
with pd.option_context("display.max_colwidth", 130, "display.width", 250):
    display(pd.DataFrame([{"modele": m_, "categorie et raison du choix": r_} for m_, r_ in CATEGORIE.items()]))

In [ ]:
# 1b. LES CONSIGNES : le meme texte pour TOUS les modeles (seul le modele change -> comparaison equitable)
#     V0 : la consigne du notebook 10 (reference historique).
#     V1 : la consigne « epeler » du notebook ameliorer_consigne, qui vise nos pieges : lettres doubles,
#          cursives proches, ligne vide, et l'interdiction de « corriger » un nom rare en nom courant.
CONSIGNES = {
    "V0": """Cette image montre un extrait de bulletin individuel du recensement francais.
Elle contient des lignes manuscrites precedees de leurs libelles imprimes :
"Nom :" et "Prenom :".

Transcris ce qui est ecrit a la main sur chacune, en MAJUSCULES, sans accent.

Regles :
- Ne corrige rien, n'ajoute rien, ne devine pas.
- Ne recopie pas les libelles imprimes, seulement l'ecriture manuscrite.
- Si une ligne n'est pas lisible avec certitude, mets exactement ILLISIBLE.

Reponds uniquement en JSON : {"NOM": "...", "PRENOM": "..."}""",
    "V1": """Cette image montre un extrait de bulletin individuel du recensement francais,
avec un nom et un prenom ecrits a la main apres les libelles imprimes "Nom :" et "Prenom :".

Transcris l'ecriture manuscrite LETTRE PAR LETTRE, en MAJUSCULES, sans accent.

Attention particuliere :
- Lettres doubles (SS, LL, RR, NN, TT, MM, FF, PP) : compte chaque lettre, n'en saute aucune.
- Lettres cursives qui se ressemblent : m/n, n/u, r/n, l/e, S/G, I/L, O/D. Regarde-les une par une.
- Si une ligne est vide, reponds VIDE. Si elle est ecrite mais illisible, reponds ILLISIBLE.
- Ne recopie pas les libelles imprimes ni l'exemple imprime.
- Ne corrige rien, ne remplace pas par un nom plus courant.

Avant de repondre, epelle chaque mot en separant les lettres par des tirets.

Reponds uniquement en JSON :
{"NOM_EPELE": "C-O-L-L-E-T", "NOM": "COLLET", "PRENOM_EPELE": "...", "PRENOM": "..."}""",
}
NOM_CONSIGNE = {"V0": "V0 simple", "V1": "V1 epeler"}
inconnues = [c for c in CONSIGNES_TESTEES if c not in CONSIGNES]
if inconnues:
    raise ValueError(f"Consignes inconnues : {inconnues} (disponibles : {list(CONSIGNES)})")
for k_ in CONSIGNES_TESTEES:
    print(f"--- {k_} ({NOM_CONSIGNE[k_]}) : {len(CONSIGNES[k_])} caracteres")

# Le journal des decisions (le meme texte qu'en fin de notebook), enregistre dans MLflow
JOURNAL_DECISIONS = "## Journal des décisions (pourquoi chaque choix)\n\n| # | Décision | Pourquoi | Ce qu'on aurait risqué sinon |\n|---|---|---|---|\n| 1 | **Même image pour tous** : le bloc nom + prénom, avec ses libellés imprimés, découpé une fois | seul le lecteur change : la comparaison est équitable | attribuer au modèle un écart dû à la découpe |\n| 2 | **Même consigne pour tous les modèles**, et deux consignes testées (V0 simple, V1 épeler) | on sépare l'**effet du modèle** de l'**effet de la consigne** (cellule 14) | conclure « ce modèle est meilleur » alors que c'est sa consigne |\n| 3 | **Vérité arbitrée par le SeRN**, pas le Fiqual seul | le Fiqual, c'est la saisie du PMQ, qui se trompe aussi | compter juste un modèle qui recopie une erreur du PMQ, et faux un modèle qui a raison contre lui |\n| 4 | **Mêmes lignes pour tous les lecteurs** ; un lecteur qui n'a pas lu 95 % des lignes est écarté | les comparaisons sont appariées, ligne à ligne | comparer des lecteurs sur des lignes différentes (plus ou moins faciles) |\n| 5 | **Critère principal : les erreurs silencieuses**, pas seulement les « justes » | pour remplacer le PMQ, le danger est le nom faux présenté comme sûr ; un « ILLISIBLE » part chez un humain | choisir un modèle qui lit plus mais se trompe sans prévenir |\n| 6 | **Tests de McNemar** (appariés) et **intervalles de Wilson** | sur ~320 lignes, un écart de 2 ou 3 points peut être dû au hasard | annoncer un gain qui n'existe pas |\n| 7 | **Catégories de modèles** : A phares fermés (le plafond), B ouverts plus gros que gemma4 (la piste production interne), C économiques (qualité/prix), D OCR spécialisés, R référence | chaque catégorie répond à une question différente | tester au hasard, sans pouvoir conclure |\n| 8 | **Température 0**, **version servie enregistrée** | lectures reproductibles ; les noms « latest » changent avec le temps | ne pas pouvoir refaire ou expliquer un résultat |\n| 9 | **Adaptation automatique** si un modèle refuse `temperature` ou `max_tokens` (modèles « qui réfléchissent ») ; `MAX_JETONS` large | sinon ces modèles échouent ou répondent vide | écarter à tort les modèles les plus récents |\n| 10 | **Essai sur 3 bulletins**, **cache**, **plafond de dépense** | ne jamais payer pour un nom de modèle faux, ni deux fois la même lecture | dépenser les crédits sans résultat |\n| 11 | **Alertes sans appel supplémentaire** : absent des listes de l'Insee, lettre doublée suspecte | elles coûtent zéro et visent nos pièges connus | |\n| 12 | **Alerte « désaccord avec gemma4 »** | deux lecteurs différents se trompent rarement de la même façon | laisser passer des erreurs silencieuses qu'un second regard attrape |\n| 13 | **Nos OCR jugés sur leurs propres lignes**, comparés aux autres sur ces mêmes lignes | TrOCR et PyLaia viennent d'une validation croisée : chaque ligne lue par un modèle qui ne l'a pas vue | comparer un modèle entraîné sur la ligne qu'il lit |\n| 14 | **Tout est enregistré dans MLflow** (`ENREGISTRER_TOUT`) : réglages, toutes les mesures, tous les tableaux, toutes les lectures, l'Excel détaillé ; un tag « mode » distingue test rapide et complet | traçabilité complète : chaque chiffre de réunion peut être retrouvé et refait ; les fichiers restent dans le stockage S3 du projet, comme les données du banc | ne pas pouvoir justifier un chiffre plus tard |\n| 15 | **Garde-fous de durée** : essai des modèles en parallèle (90 s maximum par appel), une ligne affichée à chaque appel, estimation de la durée avant de lancer, arrêt propre au-delà de `DUREE_MAX_MINUTES` ou si les appels échouent en série, bouton « stop » qui arrête vraiment, modèle écarté jamais retenté pour rien | une cellule ne doit jamais tourner des heures sans rien dire ; chaque lecture payée est gardée, relancer reprend là où on s'est arrêté | perdre du temps et des crédits sur un modèle trop lent ou mal nommé |\n\n**Limites à dire :** bulletins **fictifs** (le classement des modèles est instructif, les taux absolus peuvent changer\nsur une vraie campagne) ; ~320 lignes (écarts de moins de 3 à 4 points rarement significatifs) ; images envoyées à\nEden AI (banc fictif seulement pour l'instant).\n\n**Étendre à d'autres champs** (date de naissance, adresse, cases à cocher…) : même architecture. Il faut, pour chaque\nchamp, sa zone d'image, sa consigne, sa normalisation (une date ne se normalise pas comme un nom) et sa colonne dans le\nFiqual. Le notebook `date_naissance_adresse` en est la première version ; les mêmes lecteurs Eden AI s'y brancheront.\n"

In [ ]:
# 2. Le fichier des divergences arbitrees par le SeRN
def connexion_s3():
    for essai in range(2):
        with contextlib.suppress(Exception):
            s3fs.S3FileSystem.clear_instance_cache()
        fs_ = s3fs.S3FileSystem()
        try:
            fs_.ls(BUCKET)
            return fs_
        except PermissionError:
            if essai == 1:
                raise
            print("identifiants S3 refuses : copier les nouveaux depuis Onyxia > Mon compte > connexion au stockage")
            for v in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
                os.environ[v] = getpass.getpass(f"{v} : ").strip()


fs = connexion_s3()
try:
    f_div = sorted(f for f in fs.find(f"{BUCKET}/data/raw/{CAMPAGNE}/Divergences") if f.lower().endswith((".xlsx", ".xls")))
except FileNotFoundError:
    f_div = []
if not f_div:
    raise RuntimeError(f"Aucun fichier de divergences pour {CAMPAGNE} sur S3.")
morceaux = []
for f in f_div:
    with fs.open(f, "rb") as fh:
        morceaux.append(pd.read_excel(io.BytesIO(fh.read()), dtype=str))
DIV = pd.concat(morceaux, ignore_index=True).fillna("")
DIV.columns = [re.sub(r"[\s\-]+", "_", c.strip().upper()) for c in DIV.columns]
manque = [c for c in ["VARIABLE", "NOM_IMAGE", "VALEUR_PAD", "VALEUR_PMQ", "VALEUR_ATTENDUE"] if c not in DIV.columns]
if manque:
    raise RuntimeError(f"Colonnes introuvables dans le fichier des divergences : {manque}")
vals = DIV.VARIABLE.str.upper().str.strip().unique()
VARIABLE_NOM = VARIABLE_NOM or next((v for v in vals if re.fullmatch(r"(NOM|NOMI|NOM_INDIV|NOMNAIS)", v)), None)
VARIABLE_PRENOM = VARIABLE_PRENOM or next((v for v in vals if re.fullmatch(r"(PRENOMS?|PRENOMI|PRENOM_INDIV)", v)), None)
if not VARIABLE_NOM or not VARIABLE_PRENOM:
    print(DIV.VARIABLE.value_counts().head(15).to_string())
    raise RuntimeError("Variables NOM / PRENOM non reconnues : les renseigner en cellule 1.")


def cle_divergence(nom_image):
    """'/99/976/602/0407_9761_5300000184_BI.jpg' -> '9761_5300000184'."""
    m = Path(str(nom_image)).stem.split("_")
    if m and m[-1].upper() == "BI":
        m = m[:-1]
    return "_".join(m[-2:])


DIV["cle"] = DIV.NOM_IMAGE.map(cle_divergence)
DIV["champ"] = DIV.VARIABLE.str.upper().str.strip().map({VARIABLE_NOM: "NOM", VARIABLE_PRENOM: "PRENOM"})
DIV = DIV[DIV.champ.notna()].copy()
DIV["arbitree"] = DIV.VALEUR_ATTENDUE.str.strip() != ""
print(f"{len(DIV)} divergences NOM/PRENOM ({int(DIV.arbitree.sum())} arbitrees par le SeRN)")

In [ ]:
# 3. Bulletins, Fiqual (= PMQ), blocs
fichiers = [f for f in fs.find(f"{BUCKET}/data/raw/{CAMPAGNE}/BIE") if f.endswith("_BI.jpg")]
bi = pd.DataFrame({"chemin": fichiers})
m = bi.chemin.map(lambda c: Path(c).stem).str.split("_")
bi["cle"] = m.str[0] + "_" + m.str[1]
with fs.open(f"{BUCKET}/data/raw/{CAMPAGNE}/{FIQUAL[CAMPAGNE]}", "rb") as f:
    brut = pd.read_csv(f, sep="\x00", header=None, names=["ligne"], dtype=str,
                       encoding="latin-1", keep_default_na=False, engine="python")["ligne"]
ch = brut.str.split("|")
fq = pd.DataFrame(ch[ch.str[0] == "3"].tolist(), dtype=str)
fq["cle"] = fq[POS_UT].str.strip() + "_" + fq[POS_CAB].str.strip()
ref_df = fq[["cle", COLS["NOM"], COLS["PRENOM"]]].copy()
ref_df.columns = ["cle", "NOM", "PRENOM"]
APP = (bi.merge(ref_df, on="cle", how="inner").drop_duplicates("cle")
       .sort_values("cle").reset_index(drop=True))
APP = APP.head(10) if TEST_RAPIDE else APP
REF = {(r.cle, c): normaliser(getattr(r, c)) for r in APP.itertuples() for c in COLS}


import threading
_VERROUS_IMAGES = {}
_VERROU_GLOBAL = threading.Lock()


def bloc(cle):
    """Le bloc nom + prenom d'un bulletin. Sur quand plusieurs modeles le demandent EN MEME TEMPS :
    un seul le fabrique, ecriture atomique, et un fichier abime (coupure precedente) est refait."""
    fb = BLOCS / f"{CAMPAGNE}_{cle}.png"
    with _VERROU_GLOBAL:
        verrou = _VERROUS_IMAGES.setdefault(cle, threading.Lock())
    with verrou:
        if fb.exists():
            try:
                im = Image.open(fb)
                im.load()
                return im
            except OSError:
                fb.unlink(missing_ok=True)
        chemin = APP.loc[APP.cle == cle, "chemin"].iloc[0]
        with fs.open(chemin, "rb") as src:
            page = Image.open(io.BytesIO(src.read())).convert("RGB")
        L, H = page.size
        x, y, w, hh = ZONE_BLOC
        p = page.crop((int(x * L), int(y * H), int((x + w) * L), int((y + hh) * H)))
        tmp = fb.with_suffix(f".{threading.get_ident()}.tmp")
        p.save(tmp, format="PNG")
        os.replace(tmp, fb)
        return p


manquants = [c for c in APP.cle if not (BLOCS / f"{CAMPAGNE}_{c}.png").exists()]
if manquants:
    print(f"Decoupe de {len(manquants)} blocs (une seule fois, ensuite gardes sur le disque)...", flush=True)
    t0_ = time.time()
    with ThreadPoolExecutor(8) as ex:
        for i_, _ in enumerate(ex.map(bloc, manquants), 1):
            if i_ % 50 == 0 or i_ == len(manquants):
                print(f"  {i_}/{len(manquants)} blocs | {time.time() - t0_:.0f} s", flush=True)
print(f"{len(APP)} bulletins, {len(REF)} champs du Fiqual | divergences sur nos bulletins : "
      f"{int(DIV.cle.isin(set(APP.cle)).sum())} sur {len(DIV)} | blocs dans {BLOCS}")

In [ ]:
# 4. Cles (jamais affichees ni ecrites dans le notebook)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = (lire_vault("onyxia-kv/farisazibert/LLM").get("OPENAI_API_KEY")
                                    or getpass.getpass("cle du service LLM du SSP Cloud : ").strip())
if not os.environ.get("EDENAI_API_KEY"):
    os.environ["EDENAI_API_KEY"] = (lire_vault("onyxia-kv/farisazibert/EDENAI").get("EDENAI_API_KEY")
                                    or getpass.getpass("cle Eden AI : ").strip())
if os.environ.get("EDENAI_API_KEY", "").startswith("hvs."):
    os.environ.pop("EDENAI_API_KEY")
    raise RuntimeError("La valeur saisie est un jeton VAULT (hvs.…), pas la cle Eden AI. "
                       "Copier la cle depuis Eden AI > Manage keys, puis relancer cette cellule.")
print("cle SSP Cloud :", "presente" if os.environ.get("OPENAI_API_KEY") else "ABSENTE")
print("cle Eden AI   :", "presente" if os.environ.get("EDENAI_API_KEY") else "ABSENTE")

In [ ]:
# 5. LA LECTURE COMMUNE : meme image (le bloc), meme consigne, temperature 0, pour gemma4 comme pour Eden AI.
#    - parallele (N_PARALLELE appels a la fois), progression toutes les 30 s ;
#    - cache : chaque lecture reussie est gardee -> relancer ne refacture jamais ;
#    - temps, jetons et cout gardes dans le cache ;
#    - modeles « qui reflechissent » : s'ils refusent temperature ou max_tokens, on s'adapte automatiquement ;
#    - plafond : au-dela de PLAFOND_DOLLARS (cout renvoye par Eden AI), on arrete proprement.
VERSIONS = {}          # modele demande -> version reellement servie
ADAPTATIONS = {}       # modele -> parametres retires parce que refuses (note de methode)
COUT_TOTAL = 0.0
SECONDES_VUES = {}     # modele -> secondes moyennes par appel deja observees (pour estimer les durees)


def appel(modele, image, consigne, eden=True, delai=180, tentatives=2):
    """Renvoie ({champ: texte}, reussi, infos). infos : secondes, json_ok, jetons, cout (si fournis)."""
    t = io.BytesIO()
    image.convert("RGB").save(t, format="PNG")
    b64 = base64.b64encode(t.getvalue()).decode()
    url = URL_EDEN if eden else URL_SSP
    cle_api = os.environ["EDENAI_API_KEY"] if eden else os.environ["OPENAI_API_KEY"]
    entetes = {"Authorization": f"Bearer {cle_api}", "Content-Type": "application/json"}
    if eden:
        entetes["x-edenai-metadata"] = "enabled"
    corps = {"model": modele, "stream": False, "temperature": 0, "max_tokens": MAX_JETONS,
             "messages": [{"role": "user", "content": [
                 {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                 {"type": "text", "text": consigne}]}]}
    for p_ in ADAPTATIONS.get(modele, []):
        corps.pop(p_, None)
        if p_ == "max_tokens":
            corps["max_completion_tokens"] = MAX_JETONS
    erreur, essai = "", 0
    while essai < tentatives:
        t0 = time.time()
        try:
            r = requests.post(url, headers=entetes, json=corps, timeout=delai)
            texte_err = (getattr(r, "text", "") or "").lower() if getattr(r, "status_code", 200) == 400 else ""
            if texte_err and ("temperature" in texte_err or "max_tokens" in texte_err):
                retire = "temperature" if "temperature" in texte_err and "temperature" in corps else "max_tokens"
                if retire in corps:
                    corps.pop(retire)
                    if retire == "max_tokens":
                        corps["max_completion_tokens"] = MAX_JETONS
                    ADAPTATIONS.setdefault(modele, []).append(retire)
                    print(f"  [{modele}] parametre '{retire}' refuse par le modele : retire (note dans MLflow)")
                    continue
            r.raise_for_status()
            rep = r.json()
            texte = rep["choices"][0]["message"]["content"] or ""
            m = re.search(r"\{.*\}", texte, re.S)
            try:
                d = json.loads(m.group(0)) if m else {}
            except json.JSONDecodeError:
                d = {}
            VERSIONS[modele] = rep.get("model", modele)
            usage = rep.get("usage", {}) or {}
            infos = {"secondes": time.time() - t0, "json_ok": bool(d) and all(c in d for c in COLS),
                     "jetons": usage.get("total_tokens"), "cout": rep.get("cost", usage.get("cost"))}
            return {c: str(d.get(c, "") or "") for c in COLS}, True, infos
        except Exception as e:
            reponse = getattr(e, "response", None)
            erreur = (f"{type(e).__name__} {getattr(reponse, 'status_code', '')} "
                      f"{(getattr(reponse, 'text', '') or '')[:200]}")
            essai += 1
            time.sleep(20 * essai if getattr(reponse, "status_code", None) == 429 else 3 * essai)
    return {c: "" for c in COLS}, False, {"erreur": erreur}


def nom_fichier(texte):
    return re.sub(r"[^a-z0-9]+", "_", texte.lower()).strip("_")


def nom_lecteur(modele, consigne=None):
    return f"{modele} · {consigne}" if consigne else modele


def chemin_cache(modele, consigne=None):
    """UN SEUL fichier par modele et consigne, commun au test rapide et au complet : rien n'est jamais relu."""
    return RESULTATS / f"lectures_{nom_fichier(modele)}{'_' + consigne if consigne else ''}_{CAMPAGNE}.csv"


def lire_cache(cache, modele=None, consigne=None):
    """Le cache, enrichi des lectures deja payees sous d'anciens noms (test rapide, premiere version = V0)."""
    L_ = pd.read_csv(cache, dtype={"cle": str, "champ": str, "lu": str}) if cache.exists() \
        else pd.DataFrame(columns=["cle", "champ", "lu"])
    anciens = [cache.with_name(cache.stem + "_rapide.csv")]
    if consigne == "V0" and modele:
        anciens += [RESULTATS / f"lectures_{nom_fichier(modele)}_{CAMPAGNE}.csv",
                    RESULTATS / f"lectures_{nom_fichier(modele)}_{CAMPAGNE}_rapide.csv"]
    for ancien in anciens:
        if ancien.exists() and ancien != cache:
            A_ = pd.read_csv(ancien, dtype={"cle": str, "champ": str, "lu": str})
            L_ = pd.concat([L_, A_[~A_.cle.isin(set(L_.cle))]], ignore_index=True)
            L_.to_csv(cache, index=False)
            ancien.rename(ancien.with_suffix(".fusionne"))
            print(f"  lectures deja payees reprises de {ancien.name}")
    L_["lu"] = L_["lu"].fillna("")
    return L_


def lectures(modele, consigne=None, eden=True, cles=None, lecteur=None):
    """Lectures de tous les bulletins par un modele (et une consigne) ; cache CSV ; renvoie cle, champ, lu, ..."""
    global COUT_TOTAL
    cles = list(APP.cle) if cles is None else cles
    nom = nom_lecteur(modele, consigne)
    cache = chemin_cache(modele, consigne)
    L_ = lire_cache(cache, modele, consigne)
    faire = [c for c in cles if c not in set(L_.cle)]
    print(f"{nom} : {len(cles) - len(faire)} deja lus, {len(faire)} a lire", flush=True)
    if faire and eden and COUT_TOTAL > PLAFOND_DOLLARS:
        print(f"  PLAFOND atteint ({COUT_TOTAL:.2f} $ > {PLAFOND_DOLLARS} $) : lecture non lancee")
        return L_[L_.cle.isin(set(cles))]
    n_par = N_PARALLELE_EDEN if eden else N_PARALLELE
    sec = pd.to_numeric(L_["secondes"], errors="coerce").dropna() if "secondes" in L_ else pd.Series(dtype=float)
    moy = sec.mean() if len(sec) else SECONDES_VUES.get(modele)
    if faire:
        estim = (f"~{len(faire) * moy / n_par / 60:.0f} min ({moy:.0f} s par appel, {n_par} a la fois)"
                 if moy is not None and moy == moy else "inconnue (pas encore de lecture)")
        print(f"  duree estimee : {estim} | arret automatique au-dela de {DUREE_MAX_MINUTES} min "
              f"(ce qui est lu est garde)", flush=True)
    rows, echecs, temps, k = [], [], [], 0
    t_debut = dernier = time.time()

    stop = threading.Event()        # leve a l'interruption : les appels pas encore partis ne partent pas

    def une(cle):
        if stop.is_set():
            return cle, {}, False, {"erreur": "arrete"}
        try:
            res, ok, infos = lecteur(cle) if lecteur else appel(modele, bloc(cle), CONSIGNES[consigne], eden=eden)
        except Exception as e:      # ex. image abimee : compte comme un echec, n'arrete pas tout
            res, ok, infos = {}, False, {"erreur": f"{type(e).__name__} : {str(e)[:150]}"}
        return cle, res, ok, infos

    arret = ""
    ex = ThreadPoolExecutor(n_par)
    en_cours = {ex.submit(une, c) for c in faire}
    try:
        while en_cours:
            finis, en_cours = wait(en_cours, timeout=30, return_when=FIRST_COMPLETED)
            for f in finis:
                k += 1
                cle, res, ok, infos = f.result()
                if ok:   # temps, jetons et cout sur la ligne NOM (un appel = un bulletin)
                    rows += [{"cle": cle, "champ": c, "lu": res[c],
                              **({"secondes": round(infos["secondes"], 2), "jetons": infos["jetons"],
                                  "cout": infos["cout"]} if i == 0 else {})}
                             for i, c in enumerate(COLS)]
                    temps.append(infos["secondes"])
                    if eden and isinstance(infos.get("cout"), (int, float)):
                        COUT_TOTAL += infos["cout"]
                else:
                    echecs.append((cle, infos.get("erreur", "")))
            if not arret:
                if eden and COUT_TOTAL > PLAFOND_DOLLARS:
                    arret = f"PLAFOND de depense atteint ({COUT_TOTAL:.2f} $ > {PLAFOND_DOLLARS} $)"
                elif k >= 5 and len(echecs) == k:
                    arret = "les 5 premiers appels ont tous echoue (nom du modele, cle, ou service indisponible)"
                elif k >= 20 and len(echecs) > k / 2:
                    arret = f"plus de la moitie des appels echouent ({len(echecs)}/{k})"
                elif (time.time() - t_debut) / 60 > DUREE_MAX_MINUTES:
                    arret = f"duree maximale atteinte ({DUREE_MAX_MINUTES} min)"
                if arret and en_cours:
                    stop.set()
                    for f in en_cours:
                        f.cancel()
                    en_cours = {f for f in en_cours if not f.cancelled()}
                    print(f"  ARRET de {nom} : {arret}. Ce qui est lu est garde ; relancer la cellule reprend "
                          f"la ou on s'est arrete.", flush=True)
            if len(rows) >= 10 * len(COLS) or not en_cours:
                L_ = pd.concat([L_, pd.DataFrame(rows)], ignore_index=True)
                L_.to_csv(cache, index=False)
                rows = []
            if time.time() - dernier >= 30 or not en_cours:
                dernier = time.time()
                ecoule = time.time() - t_debut
                vitesse = f"{sum(temps[-20:]) / len(temps[-20:]):.1f} s/appel" if temps else "aucune reponse encore"
                print(f"  {nom} : {k}/{len(faire)} | {ecoule / 60:.1f} min | {vitesse} | echecs {len(echecs)} "
                      f"| cout cumule {COUT_TOTAL:.2f} $", flush=True)
    finally:
        if rows:
            L_ = pd.concat([L_, pd.DataFrame(rows)], ignore_index=True)
            L_.to_csv(cache, index=False)
        stop.set()
        ex.shutdown(wait=False, cancel_futures=True)   # bouton carre : le notebook s'arrete tout de suite
    if temps:
        SECONDES_VUES[modele] = sum(temps) / len(temps)
    if echecs:
        print(f"  {len(echecs)} echecs (relancer la cellule pour les retenter). Premier : {echecs[0][1][:200]}")
    return L_[L_.cle.isin(set(cles))]


def bilan_lectures():
    """Dit clairement si tout est lu, ou ce qu'il faut relancer (relancer ne repaie rien)."""
    incomplets = {n_: d.cle.nunique() for n_, d in LECT.items() if d.cle.nunique() < len(APP)}
    if not incomplets:
        print(f"\nCOMPLET : {len(LECT)} lecteurs, {len(APP)} bulletins chacun.")
    else:
        print("\nINCOMPLET (arret, echecs ou plafond) : relancer CETTE cellule, elle reprend la ou elle s'est arretee :")
        for n_, k_ in incomplets.items():
            print(f"  - {n_} : {k_}/{len(APP)}")


print("outils de lecture prets")

## 7. Essai : 3 bulletins par modèle, avant toute dépense

Chaque modèle lit 3 bulletins avec la consigne la plus exigeante (V1). On vérifie :

- **`json_ok`** = 1 : il répond au bon format (sinon ses lectures seraient comptées vides) ;
- **`version_servie`** : le modèle réellement utilisé (les noms « latest » changent avec le temps) ;
- **temps, jetons, coût** : l'estimation du coût total est affichée avant de lancer les 169 bulletins.

Un modèle dont le nom est faux est **écarté sans rien coûter** : corriger son nom en cellule 1 (Eden AI > Models).
L'étape 9 lance ensuite **automatiquement** les modèles qui ont réussi.

In [ ]:
# 7. ESSAI sur N_ESSAI bulletins par modele (consigne V1, la plus longue : si elle passe, V0 passe aussi)
#    - les modeles sont essayes EN MEME TEMPS (un fil par modele) : la cellule dure le temps du modele le plus lent,
#      pas la somme de tous ;
#    - une ligne s'affiche A CHAQUE appel, et toutes les 30 s on voit qui on attend encore : la cellule ne parait
#      jamais bloquee ;
#    - un appel qui depasse DELAI_ESSAI secondes est un echec : modele ecarte (trop lent pour des dizaines de milliers de bulletins) ;
#    - AUCUN GASPILLAGE : chaque lecture payee est ecrite tout de suite dans le cache des lectures. Si on
#      interrompt la cellule (bouton carre) et qu'on la relance, rien n'est repaye ; et ces documents ne seront
#      pas relus a la cellule 9.
DELAI_ESSAI = 90                    # secondes maximum par appel pendant l'essai
REESSAYER_ECARTES = False           # True = retenter les modeles ecartes a un essai precedent (ex. nom corrige)
f_essai = RESULTATS / f"essai_{CAMPAGNE}.csv"
DEJA_ESSAI = pd.read_csv(f_essai) if f_essai.exists() else pd.DataFrame(columns=["modele", "cle"])
N_ESSAI_MAX = 169                 # l'essai sert a verifier qu'un modele REPOND bien, pas a mesurer : 3 a 5 suffisent
if N_ESSAI > N_ESSAI_MAX:
    print(f"N_ESSAI = {N_ESSAI} est trop grand pour un essai : ramene a {N_ESSAI_MAX} "
          f"(la lecture de tous les documents se fait a la cellule 9, avec ses garde-fous).")
    N_ESSAI = N_ESSAI_MAX
cles_essai = list(APP.cle[:N_ESSAI])
STOP_ESSAI = threading.Event()      # leve en cas d'interruption ou d'erreur : tous les fils s'arretent
cons_essai = "V1" if "V1" in CONSIGNES_TESTEES else CONSIGNES_TESTEES[0]


def ligne_essai(modele, cle, secondes, json_ok, jetons, cout, lus):
    return {"modele": modele, "cle": cle, "categorie": CATEGORIE[modele].split(" - ")[0],
            "version_servie": VERSIONS.get(modele, "?"), "secondes": secondes, "json_ok": json_ok,
            "jetons": jetons, "cout": cout, "justes_Fiqual": sum(normaliser(lus.get(c, "")) == REF[(cle, c)] for c in COLS)}


_ECRAN = threading.Lock()


def dire(texte):
    """Affichage depuis plusieurs fils sans que les lignes se melangent."""
    with _ECRAN:
        print(texte, flush=True)


def essayer(modele):
    """Les N_ESSAI appels d'UN modele, l'un apres l'autre ; arret au premier echec.
    Un document deja lu (cache) est repris sans appel. Renvoie (lignes, reussi, cout_nouveau)."""
    cache = chemin_cache(modele, cons_essai)
    L_ = lire_cache(cache, modele, cons_essai)
    out, cout_nouveau = [], 0.0
    for i, cle in enumerate(cles_essai, 1):
        if STOP_ESSAI.is_set():                                  # interruption : on s'arrete tout de suite
            return out, False, cout_nouveau
        if cle in set(L_.cle):                                   # deja paye : repris, 0 $
            d_ = L_[L_.cle == cle]
            r_ = d_.iloc[0]
            out.append(ligne_essai(modele, cle, pd.to_numeric(r_.get("secondes"), errors="coerce"),
                                   str(r_.get("json_ok", "True")) != "False",
                                   pd.to_numeric(r_.get("jetons"), errors="coerce"),
                                   pd.to_numeric(r_.get("cout"), errors="coerce"), dict(zip(d_.champ, d_.lu))))
            dire(f"  {modele} : bulletin {i}/{N_ESSAI} deja lu (cache, 0 $)")
            continue
        try:
            image_ = bloc(cle)
        except Exception as e:                                   # probleme d'IMAGE, pas du modele : pas ecarte
            dire(f"  {modele} : image du document {i} illisible ({type(e).__name__}) - essai interrompu, "
                 f"relancer la cellule")
            return out, False, cout_nouveau
        res, ok, infos = appel(modele, image_, CONSIGNES[cons_essai], eden=True, delai=DELAI_ESSAI, tentatives=1)
        if not ok:
            err = infos.get("erreur", "")
            raison = "TROP LENT (> {DELAI_ESSAI} s)".format(DELAI_ESSAI=DELAI_ESSAI) if "Timeout" in err else err[:200]
            dire(f"  {modele} : ECHEC au bulletin {i}/{N_ESSAI} - {raison}\n"
                  f"    -> modele ecarte (nom a verifier dans Eden AI > Models, ou modele trop lent)")
            ECARTES[modele] = raison
            return out, False, cout_nouveau
        cout = infos["cout"] if isinstance(infos.get("cout"), (int, float)) else None
        cout_nouveau += cout or 0.0
        L_ = pd.concat([L_, pd.DataFrame([{"cle": cle, "champ": c, "lu": res[c],
            **({"secondes": round(infos["secondes"], 2), "jetons": infos["jetons"], "cout": cout,
                "json_ok": infos["json_ok"]} if i_ == 0 else {})} for i_, c in enumerate(COLS)])], ignore_index=True)
        L_.to_csv(cache, index=False)                            # ecrit TOUT DE SUITE : rien ne se perd
        out.append(ligne_essai(modele, cle, infos["secondes"], infos["json_ok"], infos["jetons"], cout, res))
        dire(f"  {modele} : bulletin {i}/{N_ESSAI} lu en {infos['secondes']:.0f} s | "
              f"{'reponse au bon format' if infos['json_ok'] else 'REPONSE MAL FORMEE'} | "
              f"cout {f'{cout:.4f} $' if cout is not None else 'non fourni'}")
    return out, True, cout_nouveau


f_ecartes = RESULTATS / f"essai_ecartes_{CAMPAGNE}.json"
ECARTES = json.loads(f_ecartes.read_text()) if f_ecartes.exists() and not REESSAYER_ECARTES else {}
lignes, a_essayer = [], []
for modele in MODELES_EDEN:
    deja = DEJA_ESSAI[DEJA_ESSAI.modele == modele]
    if modele in ECARTES:
        print(f"{modele} : ecarte a un essai precedent ({ECARTES[modele][:80]}) - pas retente "
              f"(REESSAYER_ECARTES = True pour le retenter)")
    elif len(deja) >= N_ESSAI:
        lignes += deja.to_dict("records")
        VERSIONS[modele] = str(deja.version_servie.iloc[0])
        print(f"{modele} : deja essaye (resultat en cache, 0 $)")
    else:
        a_essayer.append(modele)


def sauver_essai():
    E_ = pd.DataFrame(lignes)
    if len(E_):
        pd.concat([DEJA_ESSAI[~DEJA_ESSAI.modele.isin(set(E_.modele))], E_], ignore_index=True).to_csv(f_essai, index=False)


if a_essayer:
    print(f"\nEssai de {len(a_essayer)} modeles EN MEME TEMPS, {N_ESSAI} bulletins chacun "
          f"(au plus {DELAI_ESSAI} s par appel, donc au plus ~{N_ESSAI * DELAI_ESSAI // 60 + 1} min) :", flush=True)
    t_debut = time.time()
    ex = ThreadPoolExecutor(len(a_essayer))
    en_cours = {ex.submit(essayer, m_): m_ for m_ in a_essayer}
    try:
        while en_cours:
            finis, _ = wait(set(en_cours), timeout=30, return_when=FIRST_COMPLETED)
            for f in finis:
                modele = en_cours.pop(f)
                out, ok, cout_ = f.result()
                COUT_TOTAL += cout_
                if ok:
                    lignes += out
                    sauver_essai()                               # garde au fil de l'eau
            if en_cours:
                dire(f"  ... {time.time() - t_debut:.0f} s : on attend encore {', '.join(en_cours.values())}")
    finally:
        STOP_ESSAI.set()                                         # interruption ou erreur : tous les fils s'arretent
        ex.shutdown(wait=False, cancel_futures=True)             # et le notebook n'est pas bloque
    f_ecartes.write_text(json.dumps(ECARTES, ensure_ascii=False, indent=1))
    print(f"Essai termine en {time.time() - t_debut:.0f} s | cout de l'essai {COUT_TOTAL:.2f} $")

ESSAI = pd.DataFrame(lignes)
sauver_essai()
if len(ESSAI):
    SECONDES_VUES.update(pd.to_numeric(ESSAI.secondes, errors="coerce").groupby(ESSAI.modele).mean().dropna().to_dict())
OK_ESSAI = (list(ESSAI.groupby("modele").json_ok.mean().loc[lambda x: x >= 0.6].index) if len(ESSAI) else [])
if len(ESSAI):
    with pd.option_context("display.width", 250):
        display(ESSAI.groupby(["categorie", "modele", "version_servie"]).agg(
            appels=("secondes", "size"), s_par_appel=("secondes", "mean"), json_ok=("json_ok", "mean"),
            jetons_par_appel=("jetons", "mean"), justes_Fiqual=("justes_Fiqual", "sum")).round(2))
    total, minutes = 0.0, 0.0
    for m_, g_ in ESSAI.groupby("modele"):
        reste = len(APP) * len(CONSIGNES_TESTEES) - sum(
            lire_cache(chemin_cache(m_, c_), m_, c_).cle.isin(set(APP.cle)).sum() // len(COLS) for c_ in CONSIGNES_TESTEES)
        couts = pd.to_numeric(g_.cout, errors="coerce").dropna()
        s_moy = pd.to_numeric(g_.secondes, errors="coerce").mean()
        m_min = max(0, reste) * s_moy / N_PARALLELE_EDEN / 60 if s_moy == s_moy else 0.0
        minutes += m_min
        total += couts.mean() * max(0, reste) if len(couts) else 0
        print(f"{m_:34s} reste {max(0, reste):4d} lectures a faire | cout estime "
              f"{f'{couts.mean() * max(0, reste):.2f} $' if len(couts) else 'inconnu'} | duree estimee ~{m_min:.0f} min"
              + (" (ATTENTION : plus long que DUREE_MAX_MINUTES, il sera arrete puis repris a la relance)"
                 if m_min / max(1, len(CONSIGNES_TESTEES)) > DUREE_MAX_MINUTES else ""))
    print(f"\nEstimation de la cellule 9 (seulement ce qui n'est pas encore lu) : ~{total:.2f} $ "
          f"(plafond {PLAFOND_DOLLARS} $) et ~{minutes:.0f} min ({N_PARALLELE_EDEN} appels a la fois)")
print(f"\nModeles qui ont reussi l'essai : {OK_ESSAI or 'aucun'}")
print("Etape 9 : ils seront lus sur tous les bulletins, avec chaque consigne (sauf si MODELES_RETENUS est rempli).")

In [ ]:
# 8. GEMMA4, LA REFERENCE (gratuite, interne), avec chaque consigne.
#    Lectures deja faites reutilisees (memes blocs, meme consigne) : notebook 10 (V0) et ameliorer_consigne (V1).
DEJA = {"V0": RESULTATS_10 / ("listes_insee_base.csv" if CAMPAGNE == "RG25" else f"arbitrage_{CAMPAGNE}_lectures.csv"),
        "V1": RESULTATS_10 / f"consigne_{CAMPAGNE}_V1_epeler.csv"}
LECT = {}
for cons in CONSIGNES_TESTEES:
    cache_g = chemin_cache(MODELE_GEMMA, cons)
    if not cache_g.exists() and cons in DEJA and DEJA[cons].exists():
        pd.read_csv(DEJA[cons], dtype=str).fillna("")[["cle", "champ", "lu"]].to_csv(cache_g, index=False)
        print(f"gemma4 {cons} : lectures reprises de {DEJA[cons].name}")
    LECT[nom_lecteur(MODELE_GEMMA, cons)] = lectures(MODELE_GEMMA, cons, eden=False)
print({n_: f"{d.cle.nunique()}/{len(APP)} bulletins" for n_, d in LECT.items()})
bilan_lectures()

In [ ]:
# 9. LES GRANDS MODELES sur tous les bulletins, avec chaque consigne (cache : rien n'est paye deux fois)
A_LANCER = MODELES_RETENUS or OK_ESSAI
print(f"Modeles lus sur tous les bulletins : {A_LANCER or 'aucun'}"
      + ("" if MODELES_RETENUS else "  (automatiquement : ceux qui ont reussi l'essai)"))
for modele in A_LANCER:
    for cons in CONSIGNES_TESTEES:
        LECT[nom_lecteur(modele, cons)] = lectures(modele, cons, eden=True)
print(f"\nCout cumule de ce passage (renvoye par Eden AI) : {COUT_TOTAL:.2f} $")
print({n_: f"{d.cle.nunique()}/{len(APP)} bulletins" for n_, d in LECT.items()})
bilan_lectures()

In [ ]:
# 10. OCR SPECIALISES d'Eden AI (Expert Models) : l'image du bloc -> texte brut -> NOM et PRENOM retrouves
#     Ils ne suivent pas de consigne : on cherche les libelles imprimes "Nom" et "Prenom" dans le texte rendu.
#     Format de reponse peu documente : le PREMIER appel affiche la reponse brute pour verifier l'extraction.
FICHIERS_EDEN = {}


def televerser(cle):
    if cle not in FICHIERS_EDEN:
        t = io.BytesIO()
        bloc(cle).convert("RGB").save(t, format="PNG")
        r = requests.post(URL_UPLOAD, headers={"Authorization": f"Bearer {os.environ['EDENAI_API_KEY']}"},
                          files={"file": (f"{cle}.png", t.getvalue(), "image/png")}, timeout=120)
        r.raise_for_status()
        FICHIERS_EDEN[cle] = r.json()["file_id"]
    return FICHIERS_EDEN[cle]


def textes(obj, cles_texte=("text", "raw_text", "markdown", "content", "value")):
    """Tous les morceaux de texte d'une reponse JSON, dans l'ordre."""
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, str) and k.lower() in cles_texte:
                out.append(v)
            elif isinstance(v, (dict, list)):
                out += textes(v, cles_texte)
    elif isinstance(obj, list):
        for v in obj:
            out += textes(v, cles_texte)
    return out


def extraire_nom_prenom(texte):
    """Retrouve les valeurs ecrites apres les libelles imprimes 'Nom' et 'Prenom'."""
    lignes_ = [normaliser(x) for x in re.split(r"[\r\n]+", texte) if normaliser(x)]
    res = {"NOM": "", "PRENOM": ""}
    for i, l in enumerate(lignes_):
        for champ, motif in [("PRENOM", r"^PRENOMS?\b\s*(.*)$"), ("NOM", r"^NOM\b\s*(.*)$")]:
            m = re.match(motif, l)
            if m and not res[champ]:
                v = m.group(1).strip()
                if not v and i + 1 < len(lignes_) and not re.match(r"^(NOM|PRENOMS?)\b", lignes_[i + 1]):
                    v = lignes_[i + 1]
                res[champ] = v
                break
    return res


DEJA_MONTRE = set()


def lecteur_ocr(modele):
    def lire_un(cle, tentatives=2):
        erreur = ""
        for essai in range(tentatives):
            t0 = time.time()
            try:
                r = requests.post(URL_UNIVERSAL, timeout=180,
                                  headers={"Authorization": f"Bearer {os.environ['EDENAI_API_KEY']}",
                                           "Content-Type": "application/json"},
                                  json={"model": modele, "input": {"file": televerser(cle)}})
                r.raise_for_status()
                j = r.json()
                brut = "\n".join(textes(j))
                if modele not in DEJA_MONTRE:
                    DEJA_MONTRE.add(modele)
                    print(f"  [{modele}] reponse brute (debut) : {json.dumps(j, ensure_ascii=False)[:600]}")
                if not brut:
                    raise ValueError(f"aucun texte dans la reponse (statut : {j.get('status', '?')})")
                res = extraire_nom_prenom(brut)
                VERSIONS[modele] = j["model"] if isinstance(j.get("model"), str) else modele
                return res, True, {"secondes": time.time() - t0, "json_ok": any(res.values()),
                                   "jetons": None, "cout": j.get("cost")}
            except Exception as e:
                reponse = getattr(e, "response", None)
                erreur = (f"{type(e).__name__} {getattr(reponse, 'status_code', '')} "
                          f"{(getattr(reponse, 'text', '') or str(e))[:150]}")
                time.sleep(3 * (essai + 1))
        return {c: "" for c in COLS}, False, {"erreur": erreur}
    return lire_un


for modele in OCR_EDEN:
    lire_un = lecteur_ocr(modele)
    ok_essai = True
    if len(lire_cache(chemin_cache(modele))) >= N_ESSAI:     # deja essaye et lu : aucun nouvel essai paye
        print(f"--- {modele} : deja essaye (lectures en cache, 0 $)")
        LECT[modele] = lectures(modele, lecteur=lire_un)
        continue
    if modele in ECARTES:
        print(f"--- {modele} : ecarte a un essai precedent ({ECARTES[modele][:80]}) - pas retente "
              f"(REESSAYER_ECARTES = True pour le retenter)")
        continue
    print(f"--- {modele} : essai sur {N_ESSAI} bulletins")
    for cle in list(APP.cle[:N_ESSAI]):
        res, ok, infos = lire_un(cle, tentatives=1)
        if not ok:
            print(f"  ECHEC : {infos.get('erreur', '')[:200]} -> OCR abandonne")
            ECARTES[modele] = infos.get("erreur", "")[:200]
            f_ecartes.write_text(json.dumps(ECARTES, ensure_ascii=False, indent=1))
            ok_essai = False
            break
        print(f"  {cle} : lu {res} | Fiqual NOM={REF[(cle, 'NOM')]!r} PRENOM={REF[(cle, 'PRENOM')]!r} "
              f"| {infos['secondes']:.1f} s | cout {infos['cout']}")
    if ok_essai:
        LECT[modele] = lectures(modele, lecteur=lire_un)
if not OCR_EDEN:
    print("OCR_EDEN est vide : etape sautee (ex. OCR_EDEN = ['ocr/ocr/mistral'] en cellule 1).")
print({m_: f"{d.cle.nunique()}/{len(APP)} bulletins" for m_, d in LECT.items()})
if OCR_EDEN:
    bilan_lectures()

In [ ]:
# 11. Les listes de l'Insee (alerte « absent des listes » : la ligne part chez un humain)
LISTE = {}
for f, champ in [("noms2008nat_txt.zip", "NOM"), ("prenoms-2025-nat_csv.zip", "PRENOM")]:
    with zipfile.ZipFile(LISTES / f) as z:
        brut_l = z.read(next(n for n in z.namelist() if n.lower().endswith((".txt", ".csv"))))
    for e in ("utf-8", "latin-1"):
        try:
            texte = brut_l.decode(e)
            break
        except UnicodeDecodeError:
            continue
    sep = max(["\t", ";", ","], key=texte.split("\n", 1)[0].count)
    df = pd.read_csv(io.StringIO(texte), sep=sep, dtype=str, keep_default_na=False)
    df.columns = [x.strip().upper() for x in df.columns]
    if champ == "NOM":
        col = df.columns[0]
        n = df.drop(columns=[col]).apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
        dl = pd.DataFrame({"v": df[col].map(normaliser), "n": n})
        dl = dl[(dl.v != "") & (dl.v != "AUTRES NOMS")]
    else:
        col = next(x for x in ["PRENOM", "PREUSUEL"] if x in df.columns)
        colv = next(x for x in ["VALEUR", "NOMBRE"] if x in df.columns)
        dl = pd.DataFrame({"v": df[col].map(normaliser), "n": pd.to_numeric(df[colv], errors="coerce").fillna(0)})
        dl = dl[(dl.v != "") & ~dl.v.str.contains("PRENOMS RARES")]
    LISTE[champ] = dl.groupby("v").n.sum().to_dict()


def dans_liste(v, c):
    if not v or v == "ILLISIBLE":
        return True
    return v in LISTE[c] or all(x in LISTE[c] for x in v.split())




def variante_doublee(mot):
    """Les mots obtenus en doublant une lettre, ou en retirant une lettre doublee."""
    res = set()
    for i in range(len(mot)):
        if mot[i].isalpha():
            res.add(mot[:i + 1] + mot[i] + mot[i + 1:])
        if i + 1 < len(mot) and mot[i] == mot[i + 1]:
            res.add(mot[:i] + mot[i + 1:])
    res.discard(mot)
    return res


def suspect_double(v, c):
    """Alerte : une variante a lettre doublee (ou dedoublee) existe et est bien plus portee (piege COLLET / COLET)."""
    if not v or v == "ILLISIBLE":
        return False
    for mot in v.split():
        n = LISTE[c].get(mot, 0)
        if any(LISTE[c].get(w, 0) >= max(1, n) * RATIO_DOUBLE for w in variante_doublee(mot)):
            return True
    return False


print(f"listes : {len(LISTE['NOM'])} noms, {len(LISTE['PRENOM'])} prenoms")

In [ ]:
# 12. LA TABLE LIGNE PAR LIGNE : PMQ, PAD, SeRN, verite arbitree, et la lecture de chaque lecteur
divx = {(r.cle, r.champ): r for r in DIV.itertuples()}
T = pd.DataFrame([{"cle": k, "champ": c, "PMQ": v} for (k, c), v in REF.items() if v != ""])
statut, pad, sern, verite = [], [], [], []
for k, c, p in zip(T.cle, T.champ, T.PMQ):
    d = divx.get((k, c))
    if d is None:
        statut.append("sans divergence"); pad.append(p); sern.append(""); verite.append(p)
    elif not d.arbitree:
        statut.append("divergence non arbitree"); pad.append(normaliser(d.VALEUR_PAD)); sern.append(""); verite.append(None)
    else:
        statut.append("divergence arbitree"); pad.append(normaliser(d.VALEUR_PAD))
        sern.append(normaliser(d.VALEUR_ATTENDUE)); verite.append(normaliser(d.VALEUR_ATTENDUE))
T["statut"], T["PAD"], T["SeRN"], T["verite_arbitree"] = statut, pad, sern, verite
T["groupe"] = T.statut.map({"sans divergence": "A", "divergence arbitree": "B", "divergence non arbitree": "C"})


def lu_norm(v):
    v = normaliser(v)
    return "" if v == "VIDE" else v


LECTEURS = {}                                    # nom du lecteur (modele · consigne) -> colonne de T
for nom, L_ in LECT.items():
    col = nom_fichier(nom.replace(MODELE_GEMMA, "gemma4"))
    lu = {(r.cle, r.champ): lu_norm(r.lu) for r in L_.itertuples()}
    serie = [lu.get((k, c)) for k, c in zip(T.cle, T.champ)]
    couverture = sum(v is not None for v in serie) / max(1, len(serie))
    if couverture < COUVERTURE_MIN:
        print(f"ECARTE : {nom} n'a lu que {couverture:.0%} des lignes (< {COUVERTURE_MIN:.0%}) : relancer ses lectures")
        continue
    LECTEURS[nom] = col
    T[col] = serie
if not LECTEURS:
    raise RuntimeError("Aucun lecteur n'a lu assez de bulletins. Relancer les cellules 8, 9 et 10 : elles reprennent "
                       "la ou elles se sont arretees, sans rien repayer. Puis relancer la suite.")
lus_partout = T[list(LECTEURS.values())].notna().all(axis=1)
if (~lus_partout).any():
    print(f"{int((~lus_partout).sum())} lignes non lues par au moins un lecteur : ecartees (memes lignes pour tous)")
T = T[lus_partout].reset_index(drop=True)


def modele_de(nom):
    return nom.split(" · ")[0]


def consigne_de(nom):
    return nom.split(" · ")[1] if " · " in nom else "-"


def ref_gemma(nom):
    """La colonne gemma4 de la meme consigne (pour l'alerte « desaccord »)."""
    return LECTEURS.get(nom_lecteur(MODELE_GEMMA, consigne_de(nom)), LECTEURS.get(nom_lecteur(MODELE_GEMMA, "V0")))


for nom, col in LECTEURS.items():
    T[f"{col}_absent"] = [not dans_liste(v, c) for v, c in zip(T[col], T.champ)]
    T[f"{col}_double"] = [suspect_double(v, c) for v, c in zip(T[col], T.champ)]
    T[f"{col}_alerte"] = T[f"{col}_absent"] | T[f"{col}_double"]
    if modele_de(nom) != MODELE_GEMMA and ref_gemma(nom):
        T[f"{col}_desaccord"] = T[col] != T[ref_gemma(nom)]
print(f"{len(T)} lignes | " + " | ".join(f"groupe {g} ({s}) : {n}" for (g, s), n in
                                         T.groupby(["groupe", "statut"]).size().items()))
print(f"{len(LECTEURS)} lecteurs : " + ", ".join(LECTEURS))

In [ ]:
# 13. TABLEAU PRINCIPAL : chaque lecteur, seul puis avec les alertes, contre la verite arbitree ET le Fiqual
#     Trois usages par lecteur :
#       - seul                       : on prend sa lecture telle quelle ;
#       - + alerte listes / doubles  : si la lecture n'existe pas dans les listes de l'Insee, ou si une variante
#                                      a lettre doublee est bien plus portee, la ligne part chez un humain (gratuit) ;
#       - + desaccord avec gemma4    : si le modele et gemma4 (meme consigne) ne lisent pas pareil -> humain.
def case(valeur, vrai, releve=False):
    if manquant(vrai):
        return "ecartee"
    if valeur == vrai:
        return "juste"
    if valeur in ("", "ILLISIBLE"):
        return "avouee"
    return "relue" if releve else "silencieuse"


USAGES = {}                                      # nom -> (lecteur, colonne lue, colonne « vers un humain » ou None)
for nom, col in LECTEURS.items():
    USAGES[nom] = (nom, col, None)
    USAGES[f"{nom} + alerte listes/doubles"] = (nom, col, f"{col}_alerte")
    if f"{col}_desaccord" in T:
        USAGES[f"{nom} + desaccord gemma4"] = (nom, col, f"{col}_desaccord")
for u, (_, col, sig) in USAGES.items():
    rel_ = T[sig].astype(bool) if sig else pd.Series(False, index=T.index)
    T[f"{u} | Fiqual"] = [case(v, t, r) for v, t, r in zip(T[col], T.PMQ, rel_)]
    T[f"{u} | arbitree"] = [case(v, t, r) for v, t, r in zip(T[col], T.verite_arbitree, rel_)]
T["PMQ | arbitree"] = [case(v, t) for v, t in zip(T.PMQ, T.verite_arbitree)]
T["PAD | arbitree"] = [case(v, t) for v, t in zip(T.PAD, T.verite_arbitree)]
pmq_ok = T.PMQ == T.verite_arbitree


def a_un_humain(u):
    _, col, sig = USAGES[u]
    h_ = T[col].isin(["", "ILLISIBLE"])
    return h_ | T[sig].astype(bool) if sig else h_


def resume(colonne, u, verite):
    s = T[colonne]
    s = s[s != "ecartee"]
    n, k = len(s), int((s == "juste").sum())
    lecteur = USAGES[u][0] if u in USAGES else None
    ligne = {"categorie": CATEGORIE.get(modele_de(lecteur), "").split(" - ")[0] if lecteur else "saisie humaine",
             "lecteur / usage": u, "consigne": consigne_de(lecteur) if lecteur else "-", "verite": verite,
             "N": n, "justes": k, "justes_%": round(k / n * 100, 1), "IC95": ic(k, n),
             "avouees_%": round((s == "avouee").mean() * 100, 1), "relues_%": round((s == "relue").mean() * 100, 1),
             "silencieuses": int((s == "silencieuse").sum()),
             "silencieuses_%": round((s == "silencieuse").mean() * 100, 1),
             "a_un_humain_%": round(a_un_humain(u).loc[s.index].mean() * 100, 1) if lecteur else None,
             "erreurs_lettre_double": None}
    if lecteur:
        col = USAGES[u][1]
        vrai = T.verite_arbitree if verite == "arbitree" else T.PMQ
        ligne["erreurs_lettre_double"] = int(sum(bool(l) and l != v and l in variante_doublee(v)
                                                 for l, v in zip(T.loc[s.index, col], vrai.loc[s.index])))
        juste = s == "juste"
        g_col = ref_gemma(lecteur)
        if g_col and g_col != col:
            a, b, p = mcnemar(juste, (T[g_col] == vrai).loc[s.index])
            ligne["vs gemma4 meme consigne (+/-)"] = f"+{a} / -{b}"
            ligne["p vs gemma4"] = round(p, 4)
        if verite == "arbitree":
            a, b, p = mcnemar(juste, pmq_ok.loc[s.index])
            ligne["vs PMQ (+/-)"] = f"+{a} / -{b}"
            ligne["p vs PMQ"] = round(p, 4)
    return ligne


lignes = []
for u in USAGES:
    lignes.append(resume(f"{u} | arbitree", u, "arbitree"))
    lignes.append(resume(f"{u} | Fiqual", u, "Fiqual"))
lignes += [resume("PMQ | arbitree", "PMQ (saisie humaine)", "arbitree"),
           resume("PAD | arbitree", "PAD (saisie humaine)", "arbitree")]
SYNTHESE = pd.DataFrame(lignes)
ordre = ["R", "A", "B", "C", "D", "saisie humaine"]
SYNTHESE["_o"] = SYNTHESE.categorie.map(lambda c: next((i for i, x in enumerate(ordre) if c.startswith(x)), 9))
SYNTHESE = SYNTHESE.sort_values(["verite", "_o", "lecteur / usage"], kind="stable").drop(columns="_o").reset_index(drop=True)
with pd.option_context("display.max_columns", 20, "display.width", 280, "display.max_rows", 200):
    print("=== Contre la VERITE ARBITREE (la mesure de reference) ===")
    display(SYNTHESE[SYNTHESE.verite == "arbitree"])
    print("\n=== Contre le Fiqual brut (ancienne mesure, pour memoire) ===")
    display(SYNTHESE[SYNTHESE.verite == "Fiqual"])
print("\nA lire : justes + avouees + relues + silencieuses = 100 %. La colonne qui compte pour remplacer le PMQ est"
      "\nsilencieuses_% (erreur presentee comme sure). a_un_humain_% = le cout humain. p < 0,05 : ecart reel (McNemar).")

In [ ]:
# 14. EFFET DE LA CONSIGNE : pour chaque modele, V1 « epeler » fait-elle mieux que V0 « simple » ?
#     Meme modele, memes lignes : seul le texte de la consigne change -> on mesure l'effet de la consigne seule.
arb = T.verite_arbitree.map(lambda v: not manquant(v))
EFFET = []
if {"V0", "V1"} <= set(CONSIGNES_TESTEES):
    for modele in dict.fromkeys(modele_de(n) for n in LECTEURS):
        n0, n1 = nom_lecteur(modele, "V0"), nom_lecteur(modele, "V1")
        if n0 not in LECTEURS or n1 not in LECTEURS:
            continue
        c0, c1 = LECTEURS[n0], LECTEURS[n1]
        v = T.verite_arbitree[arb]
        ok0, ok1 = T[c0][arb] == v, T[c1][arb] == v
        sil0 = ~ok0 & ~T[c0][arb].isin(["", "ILLISIBLE"])
        sil1 = ~ok1 & ~T[c1][arb].isin(["", "ILLISIBLE"])
        dbl = lambda col: int(sum(bool(l) and l != t and l in variante_doublee(t) for l, t in zip(T[col][arb], v)))
        a, b, p = mcnemar(ok1, ok0)
        EFFET.append({"categorie": CATEGORIE.get(modele, "").split(" - ")[0], "modele": modele,
                      "justes_% V0": round(ok0.mean() * 100, 1), "justes_% V1": round(ok1.mean() * 100, 1),
                      "silencieuses_% V0": round(sil0.mean() * 100, 1), "silencieuses_% V1": round(sil1.mean() * 100, 1),
                      "lettres doubles V0": dbl(c0), "lettres doubles V1": dbl(c1),
                      "V1 vs V0 (+/-)": f"+{a} / -{b}", "p": round(p, 4),
                      "verdict": ("V1 meilleure (ecart reel)" if p < 0.05 and a > b else
                                  "V0 meilleure (ecart reel)" if p < 0.05 and b > a else "pas d'ecart etabli")})
EFFET = pd.DataFrame(EFFET)
if len(EFFET):
    with pd.option_context("display.max_columns", 20, "display.width", 250):
        display(EFFET)
    print("A lire : la consigne V1 vise les lettres doubles et la « correction » des noms rares. Elle est utile si les"
          "\nsilencieuses ou les erreurs de lettres doubles baissent ; « pas d'ecart etabli » = on ne peut pas conclure.")
else:
    print("Une seule consigne testee (CONSIGNES_TESTEES) : pas de comparaison des consignes.")

In [ ]:
# 15. DECOMPOSITION EN TROIS GROUPES : d'ou viennent les « justes » de chaque lecteur ?
#   A = pas de divergence (PAD = PMQ) | B = divergence arbitree par le SeRN | C = non arbitree (inconnue)
taille = T.groupe.value_counts().reindex(["A", "B", "C"]).fillna(0).astype(int)
print(f"Groupe A : {taille['A']} | groupe B : {taille['B']} (le PMQ y a raison "
      f"{int(((T.groupe == 'B') & pmq_ok).sum())} fois) | groupe C : {taille['C']}\n")
dec = []
for nom in LECTEURS:
    F, A_ = T[f"{nom} | Fiqual"] == "juste", T[f"{nom} | arbitree"] == "juste"
    gA, gB, gC = T.groupe == "A", T.groupe == "B", T.groupe == "C"
    dec.append({"lecteur": nom,
                f"justes Fiqual (/ {len(T)})": int(F.sum()),
                "dont A : vraiment justes": int((F & gA).sum()),
                "dont B : confirmes par le SeRN": int((F & gB & A_).sum()),
                "dont B : en fait FAUX (copie l'erreur du PMQ)": int((F & gB & ~A_).sum()),
                "dont C : on ne sait pas": int((F & gC).sum()),
                "faux Fiqual mais JUSTES (raison contre le PMQ)": int((~F & gB & A_).sum()),
                f"justes arbitree (/ {int((T.groupe != 'C').sum())})": int(A_.sum())})
DECOMPOSITION = pd.DataFrame(dec)
with pd.option_context("display.max_columns", 12, "display.width", 250):
    display(DECOMPOSITION.set_index("lecteur").T)
for r in dec:
    v = list(r.values())
    assert v[7] == v[2] + v[3] + v[6], r["lecteur"]
print("Verification : justes arbitree = A + B confirmes + faux Fiqual mais justes. OK.")

In [ ]:
# 16. LES DIVERGENCES : qui avait raison (PMQ ou PAD), et qui chaque modele rejoint-il ?
DV = T[T.statut != "sans divergence"].copy()
DV["PMQ_juste"] = [("?" if manquant(v) else ("oui" if p == v else "non")) for p, v in zip(DV.PMQ, DV.verite_arbitree)]
DV["PAD_juste"] = [("?" if manquant(v) else ("oui" if p == v else "non")) for p, v in zip(DV.PAD, DV.verite_arbitree)]
print(f"{len(DV)} divergences PAD / PMQ sur nos lignes ({len(DV) / max(1, len(T)):.1%}).\n")
print("Qui avait raison selon le SeRN ?")
display(pd.crosstab(DV.champ, [DV.PMQ_juste.rename("PMQ juste"), DV.PAD_juste.rename("PAD juste")], margins=True))


def rejoint(x, p, a, v):
    if not manquant(v) and x == v:
        return "SeRN"
    if x == p:
        return "PMQ"
    if x == a:
        return "PAD"
    return "illisible" if x in ("", "ILLISIBLE") else "aucun"


REJOINT = []
for nom, col in LECTEURS.items():
    DV[f"{col}_rejoint"] = [rejoint(x, p, a, v) for x, p, a, v in zip(DV[col], DV.PMQ, DV.PAD, DV.verite_arbitree)]
    cpt = DV[DV.statut == "divergence arbitree"][f"{col}_rejoint"].value_counts()
    REJOINT.append({"lecteur": nom, **{k: int(cpt.get(k, 0)) for k in ["SeRN", "PMQ", "PAD", "aucun", "illisible"]}})
print("\nSur les divergences ARBITREES, chaque lecteur lit comme... (SeRN = la bonne reponse)")
display(pd.DataFrame(REJOINT).set_index("lecteur"))

In [ ]:
# 17. TYPOLOGIE DES ERREURS (verite arbitree) : quel genre d'erreurs fait chaque lecteur ?
def type_erreur(lu, vrai):
    if lu in ("", "ILLISIBLE"):
        return "illisible ou vide"
    if lu in variante_doublee(vrai):
        return "lettre double en trop ou en moins"
    if lu.replace(" ", "") == vrai.replace(" ", ""):
        return "espace ou tiret"
    mots_lu, mots_vrai = set(lu.split()), set(vrai.split())
    if mots_lu < mots_vrai or mots_vrai < mots_lu:
        return "nom compose incomplet ou en trop"
    dd = distance(lu, vrai)
    return "1 lettre" if dd == 1 else "2 lettres" if dd == 2 else "autre (3 lettres ou plus)"


arb = T.verite_arbitree.map(lambda v: not manquant(v))
TYPES = {}
for nom, col in LECTEURS.items():
    e = T[arb & (T[col] != T.verite_arbitree)]
    TYPES[nom] = pd.Series([type_erreur(l, v) for l, v in zip(e[col], e.verite_arbitree)], dtype=str).value_counts()
TYPES = pd.DataFrame(TYPES).fillna(0).astype(int)
TYPES.loc["TOTAL"] = TYPES.sum()
display(TYPES)
print("Les erreurs d'une ou deux lettres donnent souvent un nom inexistant (rattrape par l'alerte listes) ;"
      "\nles confusions entre deux noms reels ('autre') lui echappent. Le desaccord entre deux modeles peut les attraper.")

In [ ]:
# 18. LA BARRE : la fourchette du taux d'erreur du PMQ, et ou se situe chaque lecteur / usage
n_tot = len(T)
n_err_pmq = int((T["PMQ | arbitree"] == "silencieuse").sum())
n_non_arb = int((T.statut == "divergence non arbitree").sum())
bas, haut = n_err_pmq / max(1, n_tot - n_non_arb), (n_err_pmq + n_non_arb) / max(1, n_tot)
print(f"Taux d'erreur du PMQ sur nos {n_tot} lignes : au minimum {bas:.1%} ({n_err_pmq} erreurs prouvees par le SeRN), "
      f"au maximum {haut:.1%} (si le PMQ avait tort sur les {n_non_arb} divergences non arbitrees).\n")
BARRE = SYNTHESE[(SYNTHESE.verite == "arbitree") & SYNTHESE["lecteur / usage"].isin(USAGES)][
    ["categorie", "lecteur / usage", "N", "justes_%", "silencieuses_%", "a_un_humain_%"]].copy()
BARRE["position"] = ["sous la barre basse du PMQ" if s / 100 <= bas else
                     "dans la fourchette du PMQ" if s / 100 <= haut else "au-dessus du PMQ"
                     for s in BARRE["silencieuses_%"]]
display(BARRE.sort_values(["silencieuses_%", "a_un_humain_%"]).reset_index(drop=True))
print("Critere : les erreurs SILENCIEUSES du lecteur, comparees au taux d'erreur du PMQ, "
      "en regardant aussi la part envoyee a un humain (le cout).")

In [ ]:
# 19. NOS OCR (resultats deja calcules, aucun nouvel appel) face a gemma4, au PMQ et aux grands modeles
#      Chacun est juge sur SES lignes (TrOCR et PyLaia : validation croisee, chaque ligne lue par un modele
#      qui ne l'a pas vue a l'entrainement ; TeleOCR : lignes de son test), avec la meme verite arbitree.
def chercher(nom):
    racine = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
    for d in [Path.cwd(), racine / "notebooks", racine, RESULTATS_10, Path.home() / "work" / "banc-htr" / "donnees",
              Path.home() / "work" / "mesure-qualite-rp" / "notebooks"]:
        if (d / nom).exists():
            return d / nom
    return None


NOS = {}                                         # nom -> {(cle, champ): lecture normalisee}
if COMPARER_NOS_OCR:
    f_lignes = chercher("lignes.csv")
    id_vers = {}
    if f_lignes:
        lg = pd.read_csv(f_lignes, dtype=str).fillna("")
        id_vers = {r.id: (r.cle, r.champ.upper()) for r in lg.itertuples()}
    for nom, fichier in [("TrOCR ajuste (validation croisee)", "cv_trocr.csv"),
                         ("PyLaia ajuste (validation croisee)", "cv_pylaia.csv")]:
        f_ = chercher(fichier)
        if not f_:
            print(f"{nom} : {fichier} introuvable")
            continue
        d = pd.read_csv(f_, dtype=str).fillna("")
        if "etat" in d.columns:
            d = d[d.etat == "apres"]
        col_lu = "lu_n" if "lu_n" in d.columns else "lu"
        if {"cle", "champ"} <= set(d.columns):
            cles_ = list(zip(d.cle, d.champ.str.upper()))
        else:
            cles_ = [id_vers.get(i) for i in d.id]
        NOS[nom] = {k: normaliser(v) for k, v in zip(cles_, d[col_lu]) if k}
        if not NOS[nom]:
            print(f"{nom} : lignes non reliees aux bulletins (lignes.csv introuvable ?)")
            NOS.pop(nom)
    for f_ in sorted(RESULTATS_10.glob("teleocr11_lectures_*.csv")):
        d = pd.read_csv(f_, dtype=str).fillna("")
        d = d[d.version != "gemma4"]
        for (version, entree), g_ in d.groupby(["version", "entree"]):
            NOS[f"TeleOCR {version} ({entree})"] = {(k, c.upper()): normaliser(v)
                                                     for k, c, v in zip(g_.cle, g_.champ, g_.lu)}

NOS_OCR = pd.DataFrame()
if NOS:
    vrai = dict(zip(zip(T.cle, T.champ), T.verite_arbitree))
    autres = {nom: dict(zip(zip(T.cle, T.champ), T[col])) for nom, col in LECTEURS.items()}
    lignes = []
    for nom, lu in NOS.items():
        k_ = [k for k in lu if k in vrai and not manquant(vrai[k])]
        if not k_:
            continue
        v = pd.Series([vrai[k] for k in k_])
        x = pd.Series([lu[k] for k in k_])
        juste = x == v
        avoue = x.isin(["", "ILLISIBLE"])
        ligne = {"OCR": nom, "lignes": len(k_), "justes_%": round(juste.mean() * 100, 1),
                 "IC95": ic(int(juste.sum()), len(k_)), "silencieuses_%": round((~juste & ~avoue).mean() * 100, 1),
                 "PMQ justes_% (memes lignes)": round((pd.Series([REF.get(k, "") for k in k_]) == v).mean() * 100, 1)}
        for nom_l, d_ in autres.items():
            y = pd.Series([d_.get(k, "") for k in k_]) == v
            a, b, p = mcnemar(y, juste)
            ligne[f"{nom_l} justes_% (memes lignes)"] = round(y.mean() * 100, 1)
            ligne[f"{nom_l} vs cet OCR (+/-, p)"] = f"+{a}/-{b}, p={p:.3f}"
        lignes.append(ligne)
    NOS_OCR = pd.DataFrame(lignes)
    with pd.option_context("display.max_columns", 30, "display.width", 260):
        display(NOS_OCR)
    print("Chaque OCR est juge sur ses propres lignes ; les colonnes de droite donnent, SUR CES MEMES LIGNES, le score"
          "\nde gemma4, du PMQ et de chaque grand modele, et le test de McNemar (+ = lignes justes pour le modele seul).")
else:
    print("Aucun resultat de nos OCR trouve (cv_trocr.csv, cv_pylaia.csv, teleocr11_lectures_*.csv) : etape sautee.")

In [ ]:
# 20. TEMPS ET COUT par lecteur (mesures pendant la lecture, gardees dans le cache), et projection annuelle
TEMPS = []
for modele, L_ in LECT.items():
    s_ = pd.to_numeric(L_.get("secondes"), errors="coerce").dropna() if "secondes" in L_ else pd.Series(dtype=float)
    c_ = pd.to_numeric(L_.get("cout"), errors="coerce").dropna() if "cout" in L_ else pd.Series(dtype=float)
    j_ = pd.to_numeric(L_.get("jetons"), errors="coerce").dropna() if "jetons" in L_ else pd.Series(dtype=float)
    TEMPS.append({"categorie": CATEGORIE.get(modele_de(modele), "").split(" - ")[0], "lecteur": modele,
                  "version": VERSIONS.get(modele_de(modele), modele_de(modele)), "appels_mesures": len(s_),
                  "s_par_bulletin": round(s_.mean(), 2) if len(s_) else None,
                  "min_pour_1000_bulletins (en parallele)": round(s_.mean() * 1000 / N_PARALLELE / 60, 1) if len(s_) else None,
                  "jetons_par_bulletin": round(j_.mean()) if len(j_) else None,
                  "cout_total": round(c_.sum(), 3) if len(c_) else None,
                  "cout_pour_1000_bulletins": round(c_.mean() * 1000, 2) if len(c_) else None,
                  "cout_par_an (15 lots x 2 450)": round(c_.mean() * DOCUMENTS_PAR_AN) if len(c_) else None})
TEMPS = pd.DataFrame(TEMPS)
display(TEMPS)
if TEMPS.s_par_bulletin.isna().any():
    print("Temps vide : lectures faites avant que le cache garde les durees (relire sans cache pour les mesurer).")
if TEMPS.cout_total.isna().any():
    print("Cout vide : modele interne (gratuit) ou cout non renvoye par Eden AI (voir Eden AI > Monitoring).")

In [ ]:
# 21. LE CLASSEMENT FINAL ET LES PHRASES POUR LA REUNION (calcules a partir des tableaux ci-dessus)
SEUIL_HUMAIN = 20.0        # % de lignes envoyees a un humain juge « acceptable » pour designer le meilleur compromis
arbi = SYNTHESE[SYNTHESE.verite == "arbitree"].set_index("lecteur / usage")
tps = TEMPS.set_index("lecteur") if len(TEMPS) else pd.DataFrame()
CLASSEMENT = []
for nom in LECTEURS:
    seul = arbi.loc[nom]
    combis = arbi.loc[[u for u in USAGES if USAGES[u][0] == nom and u != nom]]
    ok_h = combis[combis["a_un_humain_%"] <= SEUIL_HUMAIN]
    meilleure = (ok_h if len(ok_h) else combis).sort_values(["silencieuses_%", "a_un_humain_%"]).iloc[0] \
        if len(combis) else None
    CLASSEMENT.append({
        "categorie": seul.categorie, "lecteur": nom, "consigne": seul.consigne,
        "justes_%": seul["justes_%"], "IC95": seul.IC95, "silencieuses_% (seul)": seul["silencieuses_%"],
        "vs gemma4": (f"{seul.get('vs gemma4 meme consigne (+/-)')} (p={seul.get('p vs gemma4')})"
                      if pd.notna(seul.get("p vs gemma4", float("nan"))) else "reference"),
        "meilleure combinaison": meilleure.name.replace(nom, "").strip(" +") if meilleure is not None else "",
        "silencieuses_% (combinaison)": meilleure["silencieuses_%"] if meilleure is not None else None,
        "a_un_humain_% (combinaison)": meilleure["a_un_humain_%"] if meilleure is not None else None,
        "s_par_bulletin": tps.loc[nom, "s_par_bulletin"] if nom in tps.index else None,
        "cout_pour_1000": tps.loc[nom, "cout_pour_1000_bulletins"] if nom in tps.index else None})
CLASSEMENT = pd.DataFrame(CLASSEMENT).sort_values(["justes_%", "silencieuses_% (seul)"], ascending=[False, True])
CLASSEMENT.insert(0, "rang", range(1, len(CLASSEMENT) + 1))
with pd.option_context("display.max_columns", 20, "display.width", 280, "display.max_colwidth", 60):
    display(CLASSEMENT)

# --- Phrases pretes a dire (chiffres reels de ce passage) ---
PHRASES = []
g_lect = [n for n in LECTEURS if modele_de(n) == MODELE_GEMMA]
gem = CLASSEMENT[CLASSEMENT.lecteur.isin(g_lect)].iloc[0] if g_lect else None
autres = CLASSEMENT[~CLASSEMENT.lecteur.isin(g_lect)]
pmq = arbi.loc["PMQ (saisie humaine)"]
PHRASES.append(f"Banc {CAMPAGNE} : {len(APP)} bulletins, {len(T)} lignes nom/prenom, jugees avec la verite arbitree "
               f"par le SeRN ({int((T.groupe != 'C').sum())} lignes). Le PMQ y fait {100 - pmq['justes_%']:.1f} % "
               f"d'erreurs (fourchette {bas:.1%} - {haut:.1%}).")
if gem is not None:
    PHRASES.append(f"Reference interne : gemma4 ({gem.consigne}) lit {gem['justes_%']} % des lignes justes, "
                   f"avec {gem['silencieuses_% (seul)']} % d'erreurs silencieuses.")
if len(autres):
    b_ = autres.iloc[0]
    sig = "un ecart reel" if "p=" in str(b_["vs gemma4"]) and float(str(b_["vs gemma4"]).split("p=")[1].rstrip(")")) < 0.05 \
        else "un ecart qui n'est PAS etabli statistiquement"
    PHRASES.append(f"Meilleur lecteur teste : {b_.lecteur} ({b_.categorie}), {b_['justes_%']} % de justes {b_.IC95}, "
                   f"contre gemma4 {b_['vs gemma4']} : {sig}.")
comb = SYNTHESE[(SYNTHESE.verite == "arbitree") & SYNTHESE["lecteur / usage"].isin(USAGES)
                & (SYNTHESE["a_un_humain_%"] <= SEUIL_HUMAIN)].sort_values(["silencieuses_%", "a_un_humain_%"])
if len(comb):
    c_ = comb.iloc[0]
    pos = ("sous la barre basse du PMQ" if c_["silencieuses_%"] / 100 <= bas else
           "dans la fourchette du PMQ" if c_["silencieuses_%"] / 100 <= haut else "au-dessus du PMQ")
    PHRASES.append(f"Meilleur compromis (au plus {SEUIL_HUMAIN:.0f} % de lignes a relire) : « {c_['lecteur / usage']} » : "
                   f"{c_['silencieuses_%']} % d'erreurs silencieuses, {pos}, pour {c_['a_un_humain_%']} % de lignes "
                   f"envoyees a un humain.")
if len(EFFET):
    n_v1, n_v0 = int((EFFET.verdict.str.startswith("V1")).sum()), int((EFFET.verdict.str.startswith("V0")).sum())
    PHRASES.append(f"Consigne « epeler » (V1) contre « simple » (V0), sur {len(EFFET)} modeles : V1 meilleure pour {n_v1}, "
                   f"V0 meilleure pour {n_v0}, pas d'ecart etabli pour {len(EFFET) - n_v1 - n_v0}.")
if len(NOS_OCR):
    o_ = NOS_OCR.sort_values("justes_%", ascending=False).iloc[0]
    PHRASES.append(f"Nos OCR : le meilleur ({o_.OCR}) fait {o_['justes_%']} % sur ses {o_.lignes} lignes "
                   f"(comparaison detaillee aux grands modeles, sur les memes lignes, en cellule 19).")
PHRASES.append(f"Limites : bulletins fictifs ; {len(T)} lignes, donc un ecart de moins de 3 a 4 points n'est en general pas "
               "significatif ; les noms « latest » changent de version avec le temps (versions servies dans MLflow).")
print("\nPHRASES POUR LA REUNION :")
for p_ in PHRASES:
    print(" -", p_)

In [ ]:
# 22. FICHIER DE DETAIL LIGNE PAR LIGNE (Excel) : toutes les lectures, verdicts, et les lignes a regarder
def verdict(v, vrai):
    if manquant(vrai):
        return "VERITE INCONNUE"
    if v == vrai:
        return "JUSTE"
    if v == "":
        return "VIDE"
    if v == "ILLISIBLE":
        return "ILLISIBLE"
    return "TROMPE"


D = T[["cle", "champ", "groupe", "statut", "PMQ", "PAD", "SeRN", "verite_arbitree"]].copy()
D["UT"], D["numero_bulletin"] = D.cle.str.split("_").str[0], D.cle.str.split("_").str[1]
for nom, col in LECTEURS.items():
    D[col] = T[col]
    D[f"{col}_verdict"] = [verdict(v, t) for v, t in zip(T[col], T.verite_arbitree)]
    D[f"{col}_absent_listes"] = T[f"{col}_absent"].map({True: "oui", False: "non"})
D["PMQ_verdict"] = [verdict(v, t) for v, t in zip(T.PMQ, T.verite_arbitree)]
cols_v = [f"{c}_verdict" for c in LECTEURS.values()]
D["lecteurs_justes"] = (D[cols_v] == "JUSTE").sum(axis=1)
D["a_regarder"] = ((D[cols_v + ["PMQ_verdict"]] != "JUSTE").any(axis=1) | (D.groupe != "A")).map({True: "oui", False: "non"})
D["verite_arbitree"] = D.verite_arbitree.map(lambda v: "(inconnue)" if manquant(v) else v)
D["image_bloc"] = [str(BLOCS / f"{CAMPAGNE}_{k}.png") for k in D.cle]
D = D.sort_values(["a_regarder", "lecteurs_justes", "groupe", "cle", "champ"],
                  ascending=[False, True, True, True, True]).reset_index(drop=True)
DETAIL = D[["a_regarder", "UT", "numero_bulletin", "champ", "groupe", "statut", "PMQ", "PAD", "SeRN", "verite_arbitree"]
           + [x for c in LECTEURS.values() for x in (c, f"{c}_verdict", f"{c}_absent_listes")]
           + ["PMQ_verdict", "lecteurs_justes", "cle", "image_bloc"]]

f_xlsx = RESULTATS / f"grands_modeles_vs_pmq_{CAMPAGNE}{SUFFIXE}.xlsx"
with pd.ExcelWriter(f_xlsx) as w:
    CLASSEMENT.to_excel(w, sheet_name="0_classement", index=False)
    pd.DataFrame({"phrases pour la reunion": PHRASES}).to_excel(w, sheet_name="0_phrases", index=False)
    if len(EFFET):
        EFFET.to_excel(w, sheet_name="1b_effet_consigne", index=False)
    SYNTHESE.to_excel(w, sheet_name="1_tableau_principal", index=False)
    BARRE.to_excel(w, sheet_name="2_barre_PMQ", index=False)
    DECOMPOSITION.to_excel(w, sheet_name="3_groupes_ABC", index=False)
    pd.DataFrame(REJOINT).to_excel(w, sheet_name="4_divergences", index=False)
    TYPES.to_excel(w, sheet_name="5_types_erreurs")
    TEMPS.to_excel(w, sheet_name="6_temps_cout", index=False)
    if len(NOS_OCR):
        NOS_OCR.to_excel(w, sheet_name="6b_nos_OCR", index=False)
    DETAIL[DETAIL.a_regarder == "oui"].to_excel(w, sheet_name="7_lignes_a_regarder", index=False)
    DETAIL.to_excel(w, sheet_name="8_toutes_les_lignes", index=False)
print(f"Excel : {f_xlsx}  (contient des lectures : reste sur le service, jamais sur Git)")
with pd.option_context("display.max_rows", 40, "display.width", 250):
    display(DETAIL[DETAIL.a_regarder == "oui"].head(40))

In [ ]:
# 22 bis. Page HTML des ERREURS, avec l'image du bloc decoupe a cote de chaque ligne
import html as _html

LARGEUR_IMAGE = 320
QUALITE_JPEG = 70
MAX_LIGNES_HTML = 200            # les lignes ou le plus de modeles se trompent en premier
MODELES_HTML = list(LECT)        # ou une selection, par exemple les 4 meilleurs du CLASSEMENT

# --- la verite : verite arbitree si elle existe (T), sinon le Fiqual (REF)
VERITE = {}
if "T" in globals() and {"cle", "champ", "verite_arbitree"} <= set(T.columns):
    for r in T.itertuples():
        v = r.verite_arbitree
        if isinstance(v, str) and v and not v.startswith("("):
            VERITE[(r.cle, r.champ)] = normaliser(v)
for k_, v in REF.items():
    VERITE.setdefault(k_, v)
SOURCE_VERITE = "verite arbitree (SeRN), sinon Fiqual" if "T" in globals() else "Fiqual"

# --- une ligne par champ : la lecture de chaque modele
LUS = {m: {(r.cle, r.champ): normaliser(r.lu) for r in LECT[m].itertuples()} for m in MODELES_HTML}
lignes_err = []
for k_ in sorted(VERITE):
    lus = {m: LUS[m].get(k_) for m in MODELES_HTML}
    if any(v is None for v in lus.values()):
        continue                                     # pas lu par tous : hors comparaison
    faux = [m for m, v in lus.items() if v != VERITE[k_]]
    if faux:
        lignes_err.append({"cle": k_[0], "champ": k_[1], "verite": VERITE[k_], "lus": lus, "n_faux": len(faux)})
lignes_err.sort(key=lambda d: (-d["n_faux"], d["cle"], d["champ"]))
AFFICHEES = lignes_err[:MAX_LIGNES_HTML]


def image64(cle):
    im = bloc(cle).convert("RGB")
    im = im.resize((LARGEUR_IMAGE, max(1, int(im.height * LARGEUR_IMAGE / im.width))))
    t = io.BytesIO()
    im.save(t, format="JPEG", quality=QUALITE_JPEG, optimize=True)
    return f'<img src="data:image/jpeg;base64,{base64.b64encode(t.getvalue()).decode()}" width="{LARGEUR_IMAGE}">'


def classe(lu, vrai):
    return "ok" if lu == vrai else ("av" if lu in ("", "ILLISIBLE") else "ko")


court = {m: str(m).split("/")[-1].replace("@eu", "") for m in MODELES_HTML}
corps, deja = [], set()
for i, d in enumerate(AFFICHEES, 1):
    img = image64(d["cle"]) if d["cle"] not in deja else "<i>(meme bloc que la ligne precedente)</i>"
    deja.add(d["cle"])
    lectures = "".join(f"<div class='{classe(v, d['verite'])}'>{_html.escape(court[m])} : "
                       f"<b>{_html.escape(v) or 'vide'}</b></div>" for m, v in d["lus"].items())
    corps.append(f"<tr><td>{i}</td><td>{img}<br><small>{_html.escape(d['cle'])}</small></td>"
                 f"<td><b>{d['champ']}</b></td><td><b>{_html.escape(d['verite'])}</b></td>"
                 f"<td>{d['n_faux']}/{len(MODELES_HTML)}</td><td>{lectures}</td></tr>")

page = f"""<!doctype html><html lang="fr"><head><meta charset="utf-8"><title>Erreurs de lecture {CAMPAGNE}</title>
<style>body{{font-family:sans-serif;margin:16px}}table{{border-collapse:collapse}}td,th{{border:1px solid #ccc;
padding:6px;vertical-align:top;font-size:14px}}th{{background:#eee;position:sticky;top:0}}.ok{{background:#c6efce}}
.ko{{background:#ffc7ce}}.av{{background:#ffeb9c}}div{{padding:1px 4px;margin:1px 0}}</style></head><body>
<h1>{CAMPAGNE} : erreurs de lecture (bulletins fictifs)</h1>
<p>{len(lignes_err)} lignes ou au moins un modele se trompe ; {len(AFFICHEES)} affichees, celles ou le plus de modeles
se trompent en premier. Verite : {SOURCE_VERITE}. Vert : juste ; rouge : trompe ; jaune : vide ou illisible.</p>
<table><tr><th>n</th><th>bloc decoupe</th><th>champ</th><th>verite</th><th>modeles faux</th><th>lectures</th></tr>
{''.join(corps)}</table></body></html>"""
f_html_erreurs = RESULTATS / f"erreurs_avec_blocs_{CAMPAGNE}{SUFFIXE}.html"
f_html_erreurs.write_text(page, encoding="utf-8")
print(f"{f_html_erreurs.name} : {len(AFFICHEES)} lignes, {f_html_erreurs.stat().st_size / 1e6:.1f} Mo")

In [ ]:
# 23. MLflow : serveur du projet (Vault) et serveur personnel.
#     Avec ENREGISTRER_TOUT : TOUT le detail (reglages, mesures, tableaux, lectures, Excel).
#     Un serveur injoignable ne bloque pas : 30 s maximum par requete, puis on passe au suivant.
os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "30")
os.environ.setdefault("MLFLOW_HTTP_REQUEST_MAX_RETRIES", "2")
VARS_MLFLOW = ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD",
               "MLFLOW_TRACKING_TOKEN", "MLFLOW_TRACKING_INSECURE_TLS")


def config_personnelle():
    try:
        brut_env = open("/proc/1/environ", "rb").read().split(b"\0")
        env = dict(x.decode(errors="ignore").split("=", 1) for x in brut_env if b"=" in x)
    except OSError:
        env = dict(os.environ)
    return {k: env[k] for k in VARS_MLFLOW if k in env}


def config_projet():
    d = lire_vault("onyxia-kv/projet-mesure-qualite-rp/MLFLOW_SECRET")
    return {k: str(v) for k, v in d.items() if k in VARS_MLFLOW}


@contextlib.contextmanager
def sur_serveur(config):
    avant, avant_uri = {k: os.environ.get(k) for k in VARS_MLFLOW}, mlflow.get_tracking_uri()
    try:
        for k in VARS_MLFLOW:
            if k in config:
                os.environ[k] = config[k]
            else:
                os.environ.pop(k, None)
        mlflow.set_tracking_uri(config["MLFLOW_TRACKING_URI"])
        yield
    finally:
        for k, v in avant.items():
            os.environ.pop(k, None) if v is None else os.environ.__setitem__(k, v)
        mlflow.set_tracking_uri(avant_uri)


f_agrege = RESULTATS / f"grands_modeles_vs_pmq_agrege_{CAMPAGNE}{SUFFIXE}.xlsx"
with pd.ExcelWriter(f_agrege) as w:            # sans les lectures ligne par ligne
    CLASSEMENT.to_excel(w, sheet_name="classement", index=False)
    pd.DataFrame({"phrases": PHRASES}).to_excel(w, sheet_name="phrases", index=False)
    if len(EFFET):
        EFFET.to_excel(w, sheet_name="effet_consigne", index=False)
    SYNTHESE.to_excel(w, sheet_name="tableau_principal", index=False)
    BARRE.to_excel(w, sheet_name="barre_PMQ", index=False)
    DECOMPOSITION.to_excel(w, sheet_name="groupes_ABC", index=False)
    TYPES.to_excel(w, sheet_name="types_erreurs")
    TEMPS.to_excel(w, sheet_name="temps_cout", index=False)
    if len(NOS_OCR):
        NOS_OCR.to_excel(w, sheet_name="nos_OCR", index=False)


def enregistrer():
    with mlflow.start_run(run_name=f"grands-modeles-vs-pmq-{CAMPAGNE}") as run:
        mlflow.set_tags({"etude": "grands-modeles-vs-pmq", "notebook": "12_grands_modeles_vs_pmq",
                         "statut": "exploratoire", "mode": "test rapide" if TEST_RAPIDE else "complet", "campagne": CAMPAGNE, "donnees": "bulletins fictifs"})
        mlflow.log_params({"bulletins": len(APP), "lignes": len(T), "modeles": ",".join(LECT)[:500],
                           "versions": json.dumps(VERSIONS)[:500], "consignes": ",".join(CONSIGNES_TESTEES),
                           "cout_total_dollars": round(COUT_TOTAL, 2), "taux_erreur_PMQ_bas": round(bas, 4),
                           "taux_erreur_PMQ_haut": round(haut, 4)})
        for _, r in SYNTHESE[SYNTHESE["lecteur / usage"].isin(USAGES)].iterrows():
            nom = nom_fichier(f"{r['lecteur / usage']}_{r['verite']}")[:80]
            mlflow.log_metrics({f"{nom}_justes_pct": float(r["justes_%"]),
                                f"{nom}_silencieuses_pct": float(r["silencieuses_%"]),
                                f"{nom}_humain_pct": float(r["a_un_humain_%"])})
        for _, r in TEMPS.iterrows():
            nom = nom_fichier(r["lecteur"])[:60]
            if pd.notna(r["s_par_bulletin"]):
                mlflow.log_metric(f"temps_{nom}_s_par_bulletin", float(r["s_par_bulletin"]))
            if pd.notna(r["cout_pour_1000_bulletins"]):
                mlflow.log_metric(f"cout_{nom}_pour_1000", float(r["cout_pour_1000_bulletins"]))
        for k_ in CONSIGNES_TESTEES:
            mlflow.log_text(CONSIGNES[k_], f"consigne_{k_}.txt")
        mlflow.log_text(JOURNAL_DECISIONS, "journal_des_decisions.md")
        mlflow.log_text("\n".join(PHRASES), "phrases_reunion.txt")
        mlflow.log_dict({"versions_servies": VERSIONS, "parametres_retires": ADAPTATIONS,
                         "categories": CATEGORIE}, "modeles.json")
        mlflow.log_artifact(str(f_agrege))
        if ENREGISTRER_TOUT:
            tout_enregistrer(f"lectures_*_{CAMPAGNE}.csv")
    return run.info.run_id[:8]


def tout_enregistrer(motif_lectures):
    """TOUT le detail dans le run MLflow : reglages, toutes les mesures, tous les tableaux, toutes les lectures,
    l'Excel detaille et l'image de reperage. (ENREGISTRER_TOUT = True en cellule 1)"""
    G = globals()
    noms_reglages = ["CAMPAGNE", "TEST_RAPIDE", "N_ESSAI", "UTILISER_EDEN", "MODELES_EDEN", "MODELES_RETENUS",
                     "OCR_EDEN", "CONSIGNES_TESTEES", "COMPARER_NOS_OCR", "PLAFOND_DOLLARS", "MAX_JETONS",
                     "COUVERTURE_MIN", "N_PARALLELE", "SEUIL_HUMAIN", "RATIO_DOUBLE", "MODELE_GEMMA", "BANDE",
                     "BORDS_BLOC", "VARIABLES_ADRESSE", "COLONNES_ADRESSE", "ADR_COLS", "TYPES_ADR", "DOC",
                     "DOCUMENTS_PAR_AN", "URL_EDEN", "URL_SSP"]
    mlflow.log_params({f"reglage_{k.lower()}": str(G[k])[:500] for k in noms_reglages if k in G})
    if "ROLES" in G:
        mlflow.log_dict({f"{t}:{i}": r_ for (t, i), r_ in ROLES.items()}, "reglages/roles_colonnes_adresse.json")
    mlflow.log_dict({k: str(G[k]) for k in noms_reglages if k in G}, "reglages/reglages.json")
    # Toutes les mesures du tableau principal (tous les lecteurs, usages, morceaux et verites)
    metriques = {}
    for _, r in SYNTHESE.iterrows():
        base = nom_fichier(f"{r.get('morceau', '')}_{r['lecteur / usage']}_{r['verite']}")[:150]
        for col, court in [("justes_%", "justes_pct"), ("silencieuses_%", "silencieuses_pct"),
                           ("avouees_%", "avouees_pct"), ("relues_%", "relues_pct"), ("a_un_humain_%", "humain_pct"),
                           ("a_1_caractere_pres_%", "a_1_car_pct"), ("p vs gemma4", "p_vs_gemma4"), ("p vs PMQ", "p_vs_pmq"),
                           ("N", "n_lignes")]:
            v = pd.to_numeric(r.get(col), errors="coerce")
            if pd.notna(v):
                metriques[f"tout_{base}_{court}"] = float(v)
    for i in range(0, len(metriques), 500):
        mlflow.log_metrics(dict(list(metriques.items())[i:i + 500]))
    # Tous les tableaux, en CSV
    dossier = RESULTATS / "mlflow_detail"
    dossier.mkdir(parents=True, exist_ok=True)
    for f in dossier.glob("*.csv"):
        f.unlink()
    for nom in ["SYNTHESE", "CLASSEMENT", "EFFET", "BARRE", "DECOMPOSITION", "REJOINT", "TYPES", "TEMPS", "ESSAI",
                "PAR_MORCEAU", "NOS_OCR", "CARTE", "DETAIL"]:
        if nom in G:
            df = G[nom] if isinstance(G[nom], pd.DataFrame) else pd.DataFrame(G[nom])
            if len(df):
                df.to_csv(dossier / f"{nom.lower()}.csv", index=not isinstance(df.index, pd.RangeIndex))
    mlflow.log_artifacts(str(dossier), "tableaux")
    # Toutes les lectures brutes (une par modele et consigne), et l'Excel detaille
    for f in sorted(RESULTATS.glob(motif_lectures)):
        mlflow.log_artifact(str(f), "lectures")
    if "f_xlsx" in G and Path(G["f_xlsx"]).exists():
        mlflow.log_artifact(str(G["f_xlsx"]), "detail")
    if "f_rep" in G and G["f_rep"] and Path(G["f_rep"]).exists():
        mlflow.log_artifact(str(G["f_rep"]), "illustrations")
    print(f"  detail complet : {len(metriques)} mesures, tableaux, lectures et Excel enregistres")


if True:   # enregistre aussi en TEST RAPIDE (tag « mode » pour les distinguer dans MLflow)
    for nom_serveur, config in [("projet", config_projet()), ("personnel", config_personnelle())]:
        if "MLFLOW_TRACKING_URI" not in config:
            print(f"MLflow {nom_serveur} : non configure, essai non enregistre")
            continue
        try:
            with sur_serveur(config):
                mlflow.set_experiment("fiqual-banc-essai")
                print(f"MLflow {nom_serveur} :", enregistrer())
        except Exception as e:
            print(f"MLflow {nom_serveur} : echec - {type(e).__name__} : {str(e)[:150]}")

In [ ]:
# 23. MLflow : serveur du projet (Vault) et serveur personnel.
#     Toujours : reglages, mesures, Excel agrege, page HTML des erreurs avec les blocs decoupes.
#     Avec ENREGISTRER_TOUT : TOUT le detail (tableaux, lectures, Excel detaille).
#     Un serveur injoignable ne bloque pas : 30 s maximum par requete, puis on passe au suivant.
os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "30")
os.environ.setdefault("MLFLOW_HTTP_REQUEST_MAX_RETRIES", "2")
VARS_MLFLOW = ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD",
               "MLFLOW_TRACKING_TOKEN", "MLFLOW_TRACKING_INSECURE_TLS")
TAILLE_MAX_MO = 9                 # au-dela, le serveur MLflow risque de refuser le fichier


def config_personnelle():
    try:
        brut_env = open("/proc/1/environ", "rb").read().split(b"\0")
        env = dict(x.decode(errors="ignore").split("=", 1) for x in brut_env if b"=" in x)
    except OSError:
        env = dict(os.environ)
    return {k: env[k] for k in VARS_MLFLOW if k in env}


def config_projet():
    d = lire_vault("onyxia-kv/projet-mesure-qualite-rp/MLFLOW_SECRET")
    return {k: str(v) for k, v in d.items() if k in VARS_MLFLOW}


@contextlib.contextmanager
def sur_serveur(config):
    avant, avant_uri = {k: os.environ.get(k) for k in VARS_MLFLOW}, mlflow.get_tracking_uri()
    try:
        for k in VARS_MLFLOW:
            if k in config:
                os.environ[k] = config[k]
            else:
                os.environ.pop(k, None)
        mlflow.set_tracking_uri(config["MLFLOW_TRACKING_URI"])
        yield
    finally:
        for k, v in avant.items():
            os.environ.pop(k, None) if v is None else os.environ.__setitem__(k, v)
        mlflow.set_tracking_uri(avant_uri)


ECHECS_ENVOI = []


def envoyer(fichier, dossier=None):
    """Envoie un fichier dans le run ; un refus (taille, type) est note mais ne bloque pas le reste."""
    f_ = Path(fichier)
    if not f_.exists():
        return
    mo = f_.stat().st_size / 1e6
    if mo > TAILLE_MAX_MO:
        print(f"  ATTENTION : {f_.name} pese {mo:.1f} Mo, le serveur risque de le refuser")
    try:
        mlflow.log_artifact(str(f_), dossier)
    except Exception as e:
        ECHECS_ENVOI.append(f"{f_.name} : {type(e).__name__} {str(e)[:120]}")


# --- l'Excel agrege (sans les lectures ligne par ligne)
f_agrege = RESULTATS / f"grands_modeles_vs_pmq_agrege_{CAMPAGNE}{SUFFIXE}.xlsx"
with pd.ExcelWriter(f_agrege) as w:
    CLASSEMENT.to_excel(w, sheet_name="classement", index=False)
    if len(EFFET):
        EFFET.to_excel(w, sheet_name="effet_consigne", index=False)
    SYNTHESE.to_excel(w, sheet_name="tableau_principal", index=False)
    BARRE.to_excel(w, sheet_name="barre_PMQ", index=False)
    DECOMPOSITION.to_excel(w, sheet_name="groupes_ABC", index=False)
    TYPES.to_excel(w, sheet_name="types_erreurs")
    TEMPS.to_excel(w, sheet_name="temps_cout", index=False)
    if len(NOS_OCR):
        NOS_OCR.to_excel(w, sheet_name="nos_OCR", index=False)

HTML_ERREURS = Path(globals()["f_html_erreurs"]) if "f_html_erreurs" in globals() else None
print("page HTML des erreurs :", f"{HTML_ERREURS.name} ({HTML_ERREURS.stat().st_size / 1e6:.1f} Mo)"
      if HTML_ERREURS and HTML_ERREURS.exists() else "absente (executer la cellule 22 bis)")


def enregistrer():
    with mlflow.start_run(run_name=f"grands-modeles-vs-pmq-{CAMPAGNE}") as run:
        mlflow.set_tags({"etude": "grands-modeles-vs-pmq", "notebook": "12_grands_modeles_vs_pmq",
                         "statut": "exploratoire", "mode": "test rapide" if TEST_RAPIDE else "complet",
                         "campagne": CAMPAGNE, "donnees": "bulletins fictifs"})
        mlflow.log_params({"bulletins": len(APP), "lignes": len(T), "modeles": ",".join(LECT)[:500],
                           "versions": json.dumps(VERSIONS)[:500], "consignes": ",".join(CONSIGNES_TESTEES),
                           "cout_total_dollars": round(COUT_TOTAL, 2), "taux_erreur_PMQ_bas": round(bas, 4),
                           "taux_erreur_PMQ_haut": round(haut, 4)})
        for _, r in SYNTHESE[SYNTHESE["lecteur / usage"].isin(USAGES)].iterrows():
            nom = nom_fichier(f"{r['lecteur / usage']}_{r['verite']}")[:80]
            for col, court_ in [("justes_%", "justes_pct"), ("silencieuses_%", "silencieuses_pct"),
                                ("a_un_humain_%", "humain_pct")]:
                v = pd.to_numeric(r.get(col), errors="coerce")
                if pd.notna(v):
                    mlflow.log_metric(f"{nom}_{court_}", float(v))
        for _, r in TEMPS.iterrows():
            nom = nom_fichier(r["lecteur"])[:60]
            if pd.notna(r["s_par_bulletin"]):
                mlflow.log_metric(f"temps_{nom}_s_par_bulletin", float(r["s_par_bulletin"]))
            if pd.notna(r["cout_pour_1000_bulletins"]):
                mlflow.log_metric(f"cout_{nom}_pour_1000", float(r["cout_pour_1000_bulletins"]))
        for k_ in CONSIGNES_TESTEES:
            mlflow.log_text(CONSIGNES[k_], f"consigne_{k_}.txt")
        mlflow.log_dict({"versions_servies": VERSIONS, "parametres_retires": ADAPTATIONS,
                         "categories": CATEGORIE}, "modeles.json")
        envoyer(f_agrege)
        if HTML_ERREURS:                                   # erreurs avec les blocs decoupes
            envoyer(HTML_ERREURS, "detail")
        if ENREGISTRER_TOUT:
            tout_enregistrer(f"lectures_*_{CAMPAGNE}.csv")
        presents = [x.path for x in mlflow.artifacts.list_artifacts(run_id=run.info.run_id, artifact_path="detail")]
    return run.info.run_id[:8], presents


def tout_enregistrer(motif_lectures):
    """TOUT le detail dans le run MLflow : reglages, toutes les mesures, tous les tableaux, toutes les lectures,
    l'Excel detaille et l'image de reperage. (ENREGISTRER_TOUT = True en cellule 1)"""
    G = globals()
    noms_reglages = ["CAMPAGNE", "TEST_RAPIDE", "N_ESSAI", "UTILISER_EDEN", "MODELES_EDEN", "MODELES_RETENUS",
                     "OCR_EDEN", "CONSIGNES_TESTEES", "COMPARER_NOS_OCR", "PLAFOND_DOLLARS", "MAX_JETONS",
                     "COUVERTURE_MIN", "N_PARALLELE", "SEUIL_HUMAIN", "RATIO_DOUBLE", "MODELE_GEMMA", "BANDE",
                     "BORDS_BLOC", "VARIABLES_ADRESSE", "COLONNES_ADRESSE", "ADR_COLS", "TYPES_ADR", "DOC",
                     "DOCUMENTS_PAR_AN", "URL_EDEN", "URL_SSP"]
    mlflow.log_params({f"reglage_{k.lower()}": str(G[k])[:500] for k in noms_reglages if k in G})
    if "ROLES" in G:
        mlflow.log_dict({f"{t}:{i}": r_ for (t, i), r_ in ROLES.items()}, "reglages/roles_colonnes_adresse.json")
    mlflow.log_dict({k: str(G[k]) for k in noms_reglages if k in G}, "reglages/reglages.json")
    # Toutes les mesures du tableau principal (tous les lecteurs, usages, morceaux et verites)
    metriques = {}
    for _, r in SYNTHESE.iterrows():
        base = nom_fichier(f"{r.get('morceau', '')}_{r['lecteur / usage']}_{r['verite']}")[:150]
        for col, court_ in [("justes_%", "justes_pct"), ("silencieuses_%", "silencieuses_pct"),
                            ("avouees_%", "avouees_pct"), ("relues_%", "relues_pct"), ("a_un_humain_%", "humain_pct"),
                            ("a_1_caractere_pres_%", "a_1_car_pct"), ("p vs gemma4", "p_vs_gemma4"), ("p vs PMQ", "p_vs_pmq"),
                            ("N", "n_lignes")]:
            v = pd.to_numeric(r.get(col), errors="coerce")
            if pd.notna(v):
                metriques[f"tout_{base}_{court_}"] = float(v)
    for i in range(0, len(metriques), 500):
        mlflow.log_metrics(dict(list(metriques.items())[i:i + 500]))
    # Tous les tableaux, en CSV
    dossier = RESULTATS / "mlflow_detail"
    dossier.mkdir(parents=True, exist_ok=True)
    for f in dossier.glob("*.csv"):
        f.unlink()
    for nom in ["SYNTHESE", "CLASSEMENT", "EFFET", "BARRE", "DECOMPOSITION", "REJOINT", "TYPES", "TEMPS", "ESSAI",
                "PAR_MORCEAU", "NOS_OCR", "CARTE", "DETAIL"]:
        if nom in G:
            df = G[nom] if isinstance(G[nom], pd.DataFrame) else pd.DataFrame(G[nom])
            if len(df):
                df.to_csv(dossier / f"{nom.lower()}.csv", index=not isinstance(df.index, pd.RangeIndex))
    mlflow.log_artifacts(str(dossier), "tableaux")
    # Toutes les lectures brutes (une par modele et consigne), l'Excel detaille, l'illustration
    for f in sorted(RESULTATS.glob(motif_lectures)):
        envoyer(f, "lectures")
    if "f_xlsx" in G and Path(G["f_xlsx"]).exists():
        envoyer(G["f_xlsx"], "detail")
    if "f_rep" in G and G["f_rep"] and Path(G["f_rep"]).exists():
        envoyer(G["f_rep"], "illustrations")
    print(f"  detail complet : {len(metriques)} mesures, tableaux, lectures et Excel enregistres")


# --- enregistrement sur les deux serveurs (aussi en TEST RAPIDE : le tag « mode » les distingue)
resultats_mlflow = {}
for nom_serveur, config in [("projet", config_projet()), ("personnel", config_personnelle())]:
    if "MLFLOW_TRACKING_URI" not in config:
        print(f"MLflow {nom_serveur} : non configure, essai non enregistre")
        continue
    ECHECS_ENVOI.clear()
    try:
        with sur_serveur(config):
            mlflow.set_experiment("fiqual-banc-essai")
            run_court, presents = enregistrer()
        resultats_mlflow[nom_serveur] = run_court
        print(f"MLflow {nom_serveur} : essai {run_court} | dans 'detail' : {presents or 'rien'}")
        for e in ECHECS_ENVOI:
            print(f"   envoi refuse : {e}")
    except Exception as e:
        resultats_mlflow[nom_serveur] = f"echec : {type(e).__name__} : {str(e)[:150]}"
        print(f"MLflow {nom_serveur} : {resultats_mlflow[nom_serveur]}")
print(resultats_mlflow)